# MARBERTv2 multi-task fine-tuning

I train one shared `UBC-NLP/MARBERTv2` encoder with independent heads for complaint category, complaint urgency, customer state, and churn risk. I use the frozen `text_norm` input and the existing train and validation files only. This notebook does not load the held-out test file and does not start five-seed training unless I explicitly enable it.

**Contents:** protocol and environment, Drive paths, read-only input checks, labels and weights, tokenization, model and Kendall loss, smoke test, optional five-seed training, validation metrics, baseline comparison, artifacts, and handoff.

## Fixed protocol

| Setting | Value |
|---|---|
| Model and source text | `UBC-NLP/MARBERTv2` at pinned revision `a2a02c120b294eec3f1298e4cb5a8b111c7e3de8`, `text_norm` |
| Seeds | 42, 1337, 2023, 7, 99, in shared-config order |
| Learning rate | 3e-5, selected before this five-seed confirmation |
| Optimizer | AdamW, weight decay 0.01 |
| Epoch budget and effective batch | 3, 16 |
| Scheduler | Linear warmup, ratio 0.1, then linear decay |
| Sequence length | 128, checked against `shared_config.json` |
| Dropout and gradient clipping | 0.1 and 1.0 |
| Precision | BF16 when supported, otherwise FP16 on CUDA; full precision on CPU |
| Class imbalance | Existing train-derived class weights, churn `at_risk` cap 20 |

For task `t`, let `L_t` be its weighted cross-entropy and `s_t = log(sigma_t^2)` be a learned log variance. I minimize `L_total = sum_t [exp(-s_t) * L_t + 0.5 * s_t]`. Each `s_t` starts at zero, so each task starts with unit uncertainty. The four values are trainable parameters in the model, included in optimizer and checkpoints. The same class order and weight file as the single-task pipeline are verified before use.

**Checkpoint selection is fixed before training:** at each epoch I compute validation macro-F1 for all four tasks and select the checkpoint with the highest unweighted mean of those four scores. An exact tie selects the earliest epoch. I apply early stopping after two consecutive epochs without improvement, with three epochs as the maximum. All selection uses validation only.

**Five-seed confirmation only:** the learning-rate sweep is disabled and is not included in this notebook. Each seed starts from the same pinned pretrained revision, with fresh task heads and uncertainty parameters. Resumable optimizer checkpoints are stored separately from loadable `model.safetensors` handoff files.

Single-task comparison values are loaded only from an explicitly configured, verified run record. If no such record is available, the comparison remains pending. The TF-IDF values are the team's approved corrected-text validation baseline. Neither comparison is a test result.

## 1. Environment and GPU diagnostics

I inspect the runtime before loading the model. The Drive setup cell installs the pinned role requirements if enabled. No training runs in this section.

In [ ]:
import importlib.util
import platform
import sys

print('Python:', sys.version.split()[0], '| platform:', platform.platform())
required_modules = ('torch', 'transformers', 'safetensors', 'numpy', 'pandas', 'sklearn', 'matplotlib')
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
print('Missing runtime modules:', missing_modules if missing_modules else 'none')
print('Colab runtime:', 'google.colab' in sys.modules or importlib.util.find_spec('google.colab') is not None)

## 2. Google Drive mount and verified path resolution

Set `PROJECT_ROOT` to the exact mounted Drive folder containing `phase2-code/` and the frozen input folder. It may be under `MyDrive` or `Shareddrives`. The train, validation, and class-weight paths are explicit and validated. I do not search for or open a test CSV.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

root = Path('/content/drive/MyDrive/SVR_NEW/training_data')

for item in sorted(root.iterdir()):
    print('📁' if item.is_dir() else '📄', item.name)

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "-r", "/content/drive/MyDrive/SVR_NEW/phase2-code/requirements/base.txt",
        "-r", "/content/drive/MyDrive/SVR_NEW/phase2-code/requirements/lama.txt",
    ],
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print("\n--- ERROR DETAILS ---")
print(result.stderr[-6000:])
print("\n--- INSTALL OUTPUT ---")
print(result.stdout[-2000:])

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/SVR_NEW/phase2-code/requirements")

for name in ["base.txt", "lama.txt"]:
    print(f"\n===== {name} =====")
    print((root / name).read_text(encoding="utf-8"))

In [ ]:
import sys
import importlib.metadata as md

print("Python:", sys.version.split()[0])

packages = [
    "pandas", "numpy", "scikit-learn",
    "matplotlib", "torch", "transformers", "datasets"
]

for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

from google.colab import drive

drive.mount('/content/drive')
# Set this to the exact project folder in MyDrive or a mounted Shared drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/SVR_NEW')
CODE_ROOT = PROJECT_ROOT / 'phase2-code'
TRAINING_CODE = CODE_ROOT / 'notebooks/training'
# Change DATA_ROOT to the exact frozen-data folder if it is stored elsewhere.
DATA_ROOT = PROJECT_ROOT / 'training_data'
SHARED_CONFIG_PATH = TRAINING_CODE / 'shared_config.json'
TRAIN_CSV = DATA_ROOT / 'clean_training_train.csv'
VALIDATION_CSV = DATA_ROOT / 'clean_training_val.csv'
CLASS_WEIGHTS_PATH = DATA_ROOT / 'class_weights.json'
OUTPUT_ROOT = PROJECT_ROOT / 'marbertv2-multitask-output'
SINGLE_TASK_REFERENCE_RUN_RECORD_PATH = None  # Set an exact verified run_record.json path, or leave pending.

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Configured project root is missing: {PROJECT_ROOT}')
required_files = (
    SHARED_CONFIG_PATH,
    TRAIN_CSV,
    VALIDATION_CSV,
    CLASS_WEIGHTS_PATH,
    TRAINING_CODE / 'marbertv2_artifacts.py',
    TRAINING_CODE / 'marbertv2_churn.py',
    CODE_ROOT / 'infra/__init__.py',
    CODE_ROOT / 'infra/shared_config.py',
    CODE_ROOT / 'infra/harness.py',
    CODE_ROOT / 'infra/metrics.py',
    CODE_ROOT / 'infra/predictions_file.py',
    CODE_ROOT / 'infra/run_record.py',
    CODE_ROOT / 'infra/experiment_config.py',
    CODE_ROOT / 'infra/seeds.py',
    CODE_ROOT / 'requirements/base.txt',
    CODE_ROOT / 'requirements/lama.txt',
)
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Configured project/data files are missing. Check PROJECT_ROOT and DATA_ROOT: {missing_files}')
if TRAIN_CSV == VALIDATION_CSV:
    raise ValueError('Train and validation paths must be different files')
if any(path.resolve() == TRAIN_CSV.resolve() for path in (VALIDATION_CSV, CLASS_WEIGHTS_PATH)):
    raise ValueError('Train, validation, and class-weight files must be separate files')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
probe = tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', prefix='.drive-write-check-',
                                    dir=OUTPUT_ROOT, delete=False)
probe_path = Path(probe.name)
try:
    with probe:
        probe.write('Drive persistence check')
    if probe_path.read_text(encoding='utf-8') != 'Drive persistence check':
        raise IOError(f'Drive write/read check failed under {OUTPUT_ROOT}')
finally:
    if probe_path.exists():
        probe_path.unlink()

INSTALL_PINNED_REQUIREMENTS = False
if INSTALL_PINNED_REQUIREMENTS:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-r', str(CODE_ROOT / 'requirements/base.txt'),
         '-r', str(CODE_ROOT / 'requirements/lama.txt')],
        check=True,
    )

print('Project root:', PROJECT_ROOT)
print('Train CSV:', TRAIN_CSV)
print('Validation CSV:', VALIDATION_CSV)
print('Weights:', CLASS_WEIGHTS_PATH)
print('Shared config:', SHARED_CONFIG_PATH)
print('Output root:', OUTPUT_ROOT)
print('No test CSV path is configured or searched.')

In [ ]:
import torch
import transformers
import datasets
import pandas
import numpy
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModel
from safetensors.torch import save_file, load_file

print("Core imports: PASS")

## 3. Read-only dataset integrity checks

I verify the frozen train and validation snapshot hashes against the existing helper's recorded hashes, then check configured row counts, split names, IDs, non-empty `text_norm`, and training-derived labels against the weights file. Class dimensions are derived from those files rather than fixed counts. I do not alter, deduplicate, or split either table.

In [ ]:
for name in ["train_frame", "validation_frame", "TASKS"]:
    print(f"{name}: {'Available' if name in globals() else 'Missing'}")

In [ ]:
import hashlib
import json
import sys

sys.path.insert(0, str(TRAINING_CODE))
sys.path.insert(0, str(CODE_ROOT))
import importlib.util
PROJECT_MODULE_PATHS = {
    'infra.shared_config': CODE_ROOT / 'infra/shared_config.py',
    'infra.experiment_config': CODE_ROOT / 'infra/experiment_config.py',
    'infra.harness': CODE_ROOT / 'infra/harness.py',
    'infra.metrics': CODE_ROOT / 'infra/metrics.py',
    'infra.predictions_file': CODE_ROOT / 'infra/predictions_file.py',
    'infra.run_record': CODE_ROOT / 'infra/run_record.py',
    'infra.seeds': CODE_ROOT / 'infra/seeds.py',
    'marbertv2_artifacts': TRAINING_CODE / 'marbertv2_artifacts.py',
    'marbertv2_churn': TRAINING_CODE / 'marbertv2_churn.py',
}
import_path_errors = {}
for name, expected_path in PROJECT_MODULE_PATHS.items():
    spec = importlib.util.find_spec(name)
    if spec is None or spec.origin is None or Path(spec.origin).resolve() != expected_path.resolve():
        import_path_errors[name] = {'expected': str(expected_path), 'resolved': None if spec is None else spec.origin}
if import_path_errors:
    raise ImportError(f'Project modules do not resolve from the configured roots: {import_path_errors}')
import numpy as np
import pandas as pd
from types import SimpleNamespace
from infra.shared_config import load_shared_config
from marbertv2_artifacts import BLOBS, verify_blob

SHARED = load_shared_config(SHARED_CONFIG_PATH)
TASKS = ('complaint_category', 'complaint_urgency', 'customer_state', 'churn_risk')
FIELDS = list(TASKS)
if (SHARED['train_rows'], SHARED['val_rows'], SHARED['test_rows']) != (3575, 1003, 518):
    raise ValueError(f'Unexpected frozen split sizes in shared config: {SHARED}')
if SHARED['split_seed'] != 42 or SHARED['experiment_seeds'] != [42, 1337, 2023, 7, 99]:
    raise ValueError('The split seed or experiment seed list differs from the frozen protocol')
if SHARED['max_seq_len'] != 128:
    raise ValueError(f'Expected the verified sequence limit 128, got {SHARED["max_seq_len"]}')
verify_blob(TRAIN_CSV, 'train')
verify_blob(VALIDATION_CSV, 'val')
verify_blob(CLASS_WEIGHTS_PATH, 'weights')
weight_source = json.loads(CLASS_WEIGHTS_PATH.read_text(encoding='utf-8'))
if weight_source.get('rows') != SHARED['train_rows']:
    raise ValueError('Class weights were not computed from the configured train row count')
WEIGHTS = weight_source['weights']
if set(WEIGHTS) != set(TASKS):
    raise ValueError(f'Unexpected task keys in class weight file: {sorted(WEIGHTS)}')
CLASSES_BY_TASK = {task: list(WEIGHTS[task]) for task in TASKS}
columns = ['audio_file', 'split', 'text_norm', *FIELDS]
train_frame = pd.read_csv(TRAIN_CSV, usecols=columns, dtype=str, keep_default_na=False)
validation_frame = pd.read_csv(VALIDATION_CSV, usecols=columns, dtype=str, keep_default_na=False)
if len(train_frame) != SHARED['train_rows'] or set(train_frame['split']) != {'train'}:
    raise ValueError('Train CSV row count or split column does not match shared config')
if len(validation_frame) != SHARED['val_rows'] or set(validation_frame['split']) != {'val'}:
    raise ValueError('Validation CSV row count or split column does not match shared config')
for split_name, frame in (('train', train_frame), ('validation', validation_frame)):
    if frame['audio_file'].duplicated().any() or not frame['audio_file'].str.strip().all():
        raise ValueError(f'{split_name} contains blank or repeated audio_file identifiers')
    if not frame['text_norm'].str.strip().all():
        raise ValueError(f'{split_name} contains empty text_norm input')
overlapping_ids = (
    set(train_frame['audio_file'])
    & set(validation_frame['audio_file'])
)

overlapping_texts = (
    set(train_frame['text_norm'])
    & set(validation_frame['text_norm'])
)

print(f'Cross-split audio filename collisions: {len(overlapping_ids)}')
print(f'Cross-split identical texts: {len(overlapping_texts)}')
print('WARNING: Validation includes previously seen texts.')
for task in TASKS:
    if set(train_frame[task]) != set(CLASSES_BY_TASK[task]):
        raise ValueError(f'{task}: train labels and weight-file labels do not match')
    allowed_validation = set(CLASSES_BY_TASK[task]) | ({'unclear'} if task == 'customer_state' else set())
    if not set(validation_frame[task]).issubset(allowed_validation):
        raise ValueError(f'{task}: validation contains labels outside the train-derived mapping')
if int((validation_frame['customer_state'] == 'unclear').sum()) != 1:
    raise ValueError('Expected exactly one validation customer_state unclear row')
INPUTS = SimpleNamespace(
    shared=SHARED, weights=WEIGHTS, classes=CLASSES_BY_TASK,
    train=train_frame.to_dict('records'), validation=validation_frame.to_dict('records'),
    truth=validation_frame[['audio_file', *FIELDS]].to_dict('records'),
)
assert tuple(TASKS) == ('complaint_category', 'complaint_urgency', 'customer_state', 'churn_risk')

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

INPUT_SHA256 = {
    'train': sha256_file(TRAIN_CSV),
    'validation': sha256_file(VALIDATION_CSV),
    'class_weights': sha256_file(CLASS_WEIGHTS_PATH),
}
print(f'Frozen inputs verified: {len(INPUTS.train)} train, {len(INPUTS.validation)} validation')
print('Columns used for model input:', 'text_norm')
print('SHA-256 digests:', INPUT_SHA256)
print('No test table was opened.')

In [ ]:
shared_texts = (
    set(train_frame["text_norm"])
    & set(validation_frame["text_norm"])
)

print("Shared unique texts:", len(shared_texts))

for text in sorted(shared_texts)[:10]:
    tr = train_frame[train_frame["text_norm"] == text]
    va = validation_frame[validation_frame["text_norm"] == text]

    print("\nText:", repr(text[:150]))
    print("Train audio IDs:", tr["audio_file"].tolist()[:3])
    print("Val audio IDs:", va["audio_file"].tolist()[:3])
    print("Train labels:", tr[list(TASKS)].head(2).to_dict("records"))
    print("Val labels:", va[list(TASKS)].head(2).to_dict("records"))

## 4. Training-derived label mappings and class weights

Class order and weights come from the verified training-only `class_weights.json`. I cross-check each mapping against labels actually present in train. The validation-only `unclear` value is not a train class; existing evaluation rules exclude that one row from customer-state scoring only. The churn weight cap is 20, as recorded by the existing training configuration. I do not use the older prose value of 91.0 as a replacement for the verified weight file.

In [ ]:
import pandas as pd
from IPython.display import display

WEIGHTS = INPUTS.weights
CHURN_WEIGHT_CAP = 20.0
weight_rows = []
for task in TASKS:
    observed_train = set(train_frame[task])
    classes = list(CLASSES_BY_TASK[task])
    if set(classes) != observed_train:
        raise ValueError(f'{task}: training-derived labels disagree with weight-file labels')
    if set(WEIGHTS[task]) != set(classes):
        raise ValueError(f'{task}: weights do not cover the verified training classes')
    for label in classes:
        raw_weight = float(WEIGHTS[task][label])
        if not np.isfinite(raw_weight) or raw_weight <= 0:
            raise ValueError(f'Invalid class weight for {task}/{label}: {raw_weight}')
        used_weight = min(raw_weight, CHURN_WEIGHT_CAP) if task == 'churn_risk' and label == 'at_risk' else raw_weight
        weight_rows.append({'task': task, 'label': label, 'train_support': int((train_frame[task] == label).sum()),
                            'saved_weight': raw_weight, 'loss_weight': used_weight})
display(pd.DataFrame(weight_rows))
print('Validation customer_state labels:', validation_frame['customer_state'].value_counts().to_dict())
if int((validation_frame['customer_state'] == 'unclear').sum()) != 1:
    raise ValueError('The verified validation exception should contain exactly one unclear row')

In [ ]:
from pathlib import Path
import json

# Persistent storage on Google Drive
OUTPUT_ROOT = Path('/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5')
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LOGS_DIR = OUTPUT_ROOT / "logs"
RESULTS_DIR = OUTPUT_ROOT / "results"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"

for folder in [
    OUTPUT_ROOT,
    CHECKPOINT_DIR,
    LOGS_DIR,
    RESULTS_DIR,
    MANIFEST_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# Verify that Google Drive is writable
test_file = OUTPUT_ROOT / "drive_write_test.txt"
test_file.write_text("Google Drive storage verified.", encoding="utf-8")

assert test_file.read_text(encoding="utf-8") == \
       "Google Drive storage verified."

test_file.unlink()

print("Google Drive storage: PASS")
print("Output root:", OUTPUT_ROOT)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Logs directory:", LOGS_DIR)
print("Results directory:", RESULTS_DIR)
print("Manifest directory:", MANIFEST_DIR)

## 5. Tokenization and dataset construction

I use one tokenizer instance and tokenize only `text_norm`. I measure untruncated lengths, report how many rows exceed 128 tokens, then build train and validation datasets with the verified label indices. No labels or transcripts are printed.

In [ ]:
from torch.utils.data import Dataset

In [ ]:
class VoicemailDataset(Dataset):
    def __init__(self, rows):
        self.encodings = tokenizer(
            [row['text_norm'] for row in rows],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
            verbose=False
        )

        self.labels = [
            {
                task: LABEL_TO_ID[task].get(row[task], -100)
                for task in TASKS
            }
            for row in rows
        ]

        self.audio_files = [
            row['audio_file'] for row in rows
        ]

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, index):
        return (
            {
                key: self.encodings[key][index]
                for key in ('input_ids', 'attention_mask')
            },
            self.labels[index]
        )

In [ ]:
from pathlib import Path

root = Path(
    "/content/drive/MyDrive/SVR_NEW/"
    "marbertv2-multitask-output/runs/"
    "exp_marbertv2-multitask-kendall-lr3e5-confirmation-v1"
)

for name in ["config.json", "checkpoints/seed_42/architecture.json"]:
    path = root / name
    print(name, "EXISTS" if path.is_file() else "MISSING")

In [ ]:
import json

for name in ["config.json", "checkpoints/seed_42/architecture.json"]:
    path = root / name

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n===== {name} =====")
    print("Available keys:", list(data.keys()))

    for key, value in data.items():
        if "label" in key.lower() or "class" in key.lower() or "task" in key.lower():
            print(f"\n{key}:")
            print(json.dumps(value, ensure_ascii=False, indent=2))

In [ ]:
print("MODEL_NAME:", globals().get("MODEL_NAME", "Not defined"))
print("tokenizer:", "Available" if "tokenizer" in globals() else "Missing")

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "UBC-NLP/MARBERTv2"
MODEL_REVISION = "a2a02c120b294eec3f1298e4cb5a8b111c7e3de8"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION
)

print("Tokenizer loaded successfully!")
print("Model:", MODEL_NAME)
print("Revision:", MODEL_REVISION)

In [ ]:
MAX_LENGTH = 128

print("MAX_LENGTH:", MAX_LENGTH)

In [ ]:
import json

architecture_path = (
    root / "checkpoints/seed_42/architecture.json"
)

with open(architecture_path, "r", encoding="utf-8") as f:
    architecture = json.load(f)

LABEL_TO_ID = architecture["label_mappings"]
TASKS = tuple(architecture["tasks"])
MAX_LENGTH = architecture["max_sequence_length"]

print("Tasks:", TASKS)
print("Max length:", MAX_LENGTH)

for task in TASKS:
    print(task, ":", len(LABEL_TO_ID[task]), "classes")

print(
    "Unclear mapping:",
    LABEL_TO_ID["customer_state"].get("unclear", -100)
)

In [ ]:
TRAIN_DATASET = VoicemailDataset(INPUTS.train)
VALIDATION_DATASET = VoicemailDataset(INPUTS.validation)
print("Train:", len(TRAIN_DATASET))
print("Validation:", len(VALIDATION_DATASET))

unclear_index = validation_frame.index[
    validation_frame["customer_state"] == "unclear"
][0]

print(
    "Unclear label:",
    VALIDATION_DATASET.labels[unclear_index]["customer_state"]
)

## 6. Shared encoder and four independent heads

The shared MARBERTv2 encoder feeds a dropout layer and one linear classifier per verified task mapping. Output dimensions are derived from the training-supported label lists, never typed as constants.

In [ ]:
import torch.nn as nn
from transformers import AutoModel

DROPOUT = 0.1

class MultiTaskMARBERT(nn.Module):
    def __init__(self, encoder, classes_by_task, dropout):
        super().__init__()
        self.encoder = encoder
        hidden_size = int(encoder.config.hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({task: nn.Linear(hidden_size, len(classes_by_task[task]))
                                    for task in classes_by_task})
        self.log_variances = nn.Parameter(torch.zeros(len(classes_by_task), dtype=torch.float32))
        self.task_order = tuple(classes_by_task)

    def forward(self, input_ids, attention_mask):
        pooled = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        shared = self.dropout(pooled)
        return {task: self.heads[task](shared) for task in self.task_order}

def load_pretrained_model():
    encoder = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    return MultiTaskMARBERT(encoder, CLASSES_BY_TASK, DROPOUT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    AMP_ENABLED = True
    RUNTIME_DEVICE = torch.cuda.get_device_name(0)
    print('GPU:', RUNTIME_DEVICE, '| AMP:', str(AMP_DTYPE))
else:
    AMP_DTYPE = None
    AMP_ENABLED = False
    RUNTIME_DEVICE = 'cpu'
    print('GPU unavailable: full-precision CPU mode. Training remains opt-in.')
print('Torch:', torch.__version__, '| Transformers:', __import__('transformers').__version__)

## 7. Kendall uncertainty-weighted loss

For each task, I apply the verified class-weighted cross-entropy, then add `exp(-s_t) * L_t + 0.5 * s_t`. The `log_variances` parameter vector is part of `MultiTaskMARBERT`, is included in AdamW with no weight decay, receives gradients, and is stored in both resumable and selected checkpoints.

In [ ]:
from contextlib import nullcontext
import math

WEIGHT_DECAY = 0.01
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
GRADIENT_CLIP_NORM = 1.0
EPOCHS = 3
EARLY_STOPPING_PATIENCE = 2
EFFECTIVE_BATCH_SIZE = 16
MICRO_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
if MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS != EFFECTIVE_BATCH_SIZE:
    raise ValueError('Micro batch times accumulation must remain the effective batch size of 16')

CLASS_WEIGHT_TENSORS = {
    task: torch.tensor([
        min(float(WEIGHTS[task][label]), CHURN_WEIGHT_CAP)
        if task == 'churn_risk' and label == 'at_risk' else float(WEIGHTS[task][label])
        for label in CLASSES_BY_TASK[task]
    ], dtype=torch.float32, device=DEVICE)
    for task in TASKS
}

def task_losses(logits, labels):
    return {
        task: nn.functional.cross_entropy(logits[task], labels[task], weight=CLASS_WEIGHT_TENSORS[task])
        for task in TASKS
    }

def kendall_total_loss(model, losses):
    terms = [torch.exp(-model.log_variances[index]) * losses[task]
             + 0.5 * model.log_variances[index]
             for index, task in enumerate(TASKS)]
    return torch.stack(terms).sum()

def build_optimizer(model):
    decay, no_decay = [], []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name == 'log_variances' or name.endswith(('bias', 'LayerNorm.weight')):
            no_decay.append(parameter)
        else:
            decay.append(parameter)
    grouped_ids = [id(parameter) for group in (decay, no_decay) for parameter in group]
    expected_ids = [id(parameter) for parameter in model.parameters() if parameter.requires_grad]
    if len(grouped_ids) != len(set(grouped_ids)) or set(grouped_ids) != set(expected_ids):
        raise ValueError('AdamW parameter groups must cover every trainable parameter exactly once')
    return torch.optim.AdamW(
        [{'params': decay, 'weight_decay': WEIGHT_DECAY}, {'params': no_decay, 'weight_decay': 0.0}],
        lr=LEARNING_RATE,
        betas=(0.9, 0.999), eps=1e-8,
    )

def autocast_context():
    return torch.autocast(device_type='cuda', dtype=AMP_DTYPE) if AMP_ENABLED else nullcontext()

## 8. One-batch smoke test

This test loads the pinned pretrained encoder, runs all four heads on a real train batch, checks finite weighted losses and gradients, and verifies that the uncertainty parameters update. It trains on one batch only and does not score validation. The five-seed loop below remains disabled.

In [ ]:
import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

if torch.cuda.get_device_capability(0)[0] < 8:
    AMP_DTYPE = torch.float16

print("Selected AMP dtype:", AMP_DTYPE)

In [ ]:
print("SMOKE_MODEL exists:", "SMOKE_MODEL" in globals())
print("smoke_scaler exists:", "smoke_scaler" in globals())
print("SMOKE_MODEL candidates:",
      [name for name in globals()
       if "model" in name.lower() or "smoke" in name.lower()])

In [ ]:
AMP_ENABLED = False
AMP_DTYPE = None

print("Diagnostic mode: FP32")
print("AMP enabled:", AMP_ENABLED)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
import inspect

print("TRAIN_DATASET exists:", "TRAIN_DATASET" in globals())
print("collate_batch exists:", "collate_batch" in globals())

if "TRAIN_DATASET" in globals():
    print("Dataset type:", type(TRAIN_DATASET))
    print("Dataset length:", len(TRAIN_DATASET))
    print("First sample:", repr(TRAIN_DATASET[0])[:1500])

print("Possible collate functions:")
print([name for name in globals() if "collate" in name.lower()])

In [ ]:
print("Tokenizer exists:", "TOKENIZER" in globals())
print("Tokenizer candidates:",
      [name for name in globals() if "tokenizer" in name.lower()])

print("Padding token ID:",
      TOKENIZER.pad_token_id if "TOKENIZER" in globals() else "Unknown")

print("Task names:", TASKS if "TASKS" in globals() else "Not defined")

print("Max length:", MAX_LENGTH if "MAX_LENGTH" in globals() else "Not defined")

In [ ]:
print("Tokenizer type:", type(tokenizer).__name__)
print("Padding token:", tokenizer.pad_token)
print("Padding token ID:", tokenizer.pad_token_id)
print("Model max length:", tokenizer.model_max_length)

In [ ]:
import torch

def collate_batch(batch):
    inputs, labels = zip(*batch)

    padded_inputs = tokenizer.pad(
        list(inputs),
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    batch_labels = {
        task: torch.tensor(
            [sample_labels[task] for sample_labels in labels],
            dtype=torch.long,
        )
        for task in TASKS
    }

    return padded_inputs, batch_labels

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    TRAIN_DATASET,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_batch,
)

test_inputs, test_labels = next(iter(test_loader))

print("Input shape:", test_inputs["input_ids"].shape)
print("Attention mask shape:", test_inputs["attention_mask"].shape)

for task in TASKS:
    print(task, test_labels[task].shape)

In [ ]:
from infra.seeds import set_global_seed
from safetensors.torch import load_file, save_file
from infra.predictions_file import read_predictions_file, write_predictions_file
import tempfile

set_global_seed(42)
SMOKE_MODEL = load_pretrained_model().to(DEVICE)
smoke_loader = DataLoader(TRAIN_DATASET, batch_size=4, shuffle=False, collate_fn=collate_batch)
smoke_inputs, smoke_labels = next(iter(smoke_loader))
smoke_inputs = {key: value.to(DEVICE) for key, value in smoke_inputs.items()}
smoke_labels = {key: value.to(DEVICE) for key, value in smoke_labels.items()}
smoke_optimizer = build_optimizer(SMOKE_MODEL)
smoke_scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED and AMP_DTYPE == torch.float16)
SMOKE_MODEL.train()
smoke_optimizer.zero_grad(set_to_none=True)
with autocast_context():
    smoke_logits = SMOKE_MODEL(**smoke_inputs)
    smoke_losses = task_losses(smoke_logits, smoke_labels)
    smoke_total = kendall_total_loss(SMOKE_MODEL, smoke_losses)
if set(smoke_logits) != set(TASKS):
    raise ValueError(f'Expected logits for all four tasks, got {sorted(smoke_logits)}')
for task in TASKS:
    expected_shape = (smoke_inputs['input_ids'].shape[0], len(CLASSES_BY_TASK[task]))
    if tuple(smoke_logits[task].shape) != expected_shape or tuple(smoke_labels[task].shape) != (expected_shape[0],):
        raise ValueError(f'{task}: logits or labels have an invalid smoke-test shape')
if not torch.isfinite(smoke_total):
    raise FloatingPointError(f'Non-finite smoke loss: {smoke_total.item()}')
smoke_scaler.scale(smoke_total).backward()
smoke_scaler.unscale_(smoke_optimizer)
torch.nn.utils.clip_grad_norm_(SMOKE_MODEL.parameters(), GRADIENT_CLIP_NORM)
if SMOKE_MODEL.log_variances.grad is None or not torch.isfinite(SMOKE_MODEL.log_variances.grad).all():
    raise RuntimeError('Uncertainty parameters did not receive finite gradients')
before_uncertainty = SMOKE_MODEL.log_variances.detach().clone()
smoke_scaler.step(smoke_optimizer)
smoke_scaler.update()
if torch.equal(before_uncertainty, SMOKE_MODEL.log_variances.detach()):
    raise RuntimeError('Smoke test did not update the uncertainty parameters')
if any(not torch.isfinite(loss.detach()) for loss in smoke_losses.values()):
    raise FloatingPointError('A task loss was non-finite')
with tempfile.TemporaryDirectory() as smoke_dir:
    model_path = Path(smoke_dir) / 'model.safetensors'
    model_state = {key: value.detach().cpu().contiguous() for key, value in SMOKE_MODEL.state_dict().items()}
    save_file(model_state, str(model_path), metadata={'model': MODEL_NAME, 'revision': MODEL_REVISION})
    loaded_state = load_file(str(model_path), device='cpu')
    if set(loaded_state) != set(model_state) or any(not torch.equal(loaded_state[key], value) for key, value in model_state.items()):
        raise RuntimeError('safetensors model-state round trip failed')
    smoke_probabilities = {task: {label: 1.0 / len(CLASSES_BY_TASK[task]) for label in CLASSES_BY_TASK[task]} for task in TASKS}
    prediction_path = Path(smoke_dir) / 'predictions_val_seed42.json'
    write_predictions_file(prediction_path, 'smoke-test', MODEL_NAME, 'val', 42,
                           SHARED['dataset_version'], [{'audio_file': 'smoke.wav', 'probabilities': smoke_probabilities}],
                           CLASSES_BY_TASK)
    smoke_prediction = read_predictions_file(prediction_path, CLASSES_BY_TASK, expected_split='val')
    if smoke_prediction['meta']['n_rows'] != 1 or set(smoke_prediction['harness_rows'][0]) != {'audio_file', *TASKS}:
        raise RuntimeError('Prediction schema smoke test failed')
print('Smoke test passed on', DEVICE, '| total loss:', float(smoke_total.detach()),
      '| initial uncertainty:', [0.0] * len(TASKS),
      '| updated uncertainty:', SMOKE_MODEL.log_variances.detach().cpu().tolist())

# del smoke_loader, smoke_inputs, smoke_labels, smoke_logits, smoke_losses, smoke_total, smoke_optimizer, smoke_scaler
# del SMOKE_MODEL
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

In [ ]:
import torch

p = SMOKE_MODEL.log_variances

print("AMP_ENABLED:", AMP_ENABLED)
print("AMP_DTYPE:", AMP_DTYPE)

print("Before:", before_uncertainty.detach().cpu().tolist())
print("After: ", p.detach().cpu().tolist())

print(
    "Max absolute change:",
    (p.detach() - before_uncertainty).abs().max().item()
)

print("Gradient exists:", p.grad is not None)

if p.grad is not None:
    print("Gradient:", p.grad.detach().cpu().tolist())
    print("Gradient finite:", torch.isfinite(p.grad).all().item())

print("Parameter dtype:", p.dtype)
print("GradScaler scale:", smoke_scaler.get_scale())

print("Optimizer parameter groups:")

for i, group in enumerate(smoke_optimizer.param_groups):
    contains_uncertainty = any(
        param is p for param in group["params"]
    )

    print({
        "group": i,
        "lr": group["lr"],
        "weight_decay": group.get("weight_decay"),
        "contains_log_variances": contains_uncertainty,
    })

In [ ]:
nonfinite = []
finite_nonzero = 0
zero_grad = 0
missing_grad = 0

for name, param in SMOKE_MODEL.named_parameters():
    if param.grad is None:
        missing_grad += 1
        continue

    grad = param.grad.detach()

    if not torch.isfinite(grad).all():
        nonfinite.append(name)
    elif torch.count_nonzero(grad).item() > 0:
        finite_nonzero += 1
    else:
        zero_grad += 1

print("Parameters with non-finite gradients:", len(nonfinite))
print("Examples:", nonfinite[:10])
print("Parameters with finite non-zero gradients:", finite_nonzero)
print("Parameters with zero gradients:", zero_grad)
print("Parameters without gradients:", missing_grad)
print("GradScaler current scale:", smoke_scaler.get_scale())

In [ ]:
ORIGINAL_AMP_ENABLED = AMP_ENABLED
AMP_ENABLED = False

print("Diagnostic AMP_ENABLED:", AMP_ENABLED)
print("Original AMP_ENABLED:", ORIGINAL_AMP_ENABLED)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

print("Drive accessible:", root.exists())

if root.exists():
    for item in sorted(root.iterdir()):
        print("📁" if item.is_dir() else "📄", item.name)

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/SVR_NEW")

print("📁 SVR_NEW contents:")
for item in sorted(root.iterdir()):
    print("📁" if item.is_dir() else "📄", item.name)

print("\n🔎 Searching for infra/seeds.py:")
matches = list(root.rglob("infra/seeds.py"))

if matches:
    for path in matches[:20]:
        print("FOUND:", path)
else:
    print("Not found in SVR_NEW")

In [ ]:
from pathlib import Path
import hashlib

root = Path("/content/drive/MyDrive/SVR_NEW")

folders = {
    "phase2-code": root / "phase2-code",
    "phase2-code-single-seed": root / "phase2-code-single-seed",
}

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

files = {}

for name, folder in folders.items():
    files[name] = {
        str(p.relative_to(folder)): sha256(p)
        for p in folder.rglob("*.py")
    }

    print(f"\n{name}")
    print("Python files:", len(files[name]))
    print("Top-level contents:")
    print([p.name for p in folder.iterdir()][:20])

common = set.intersection(*(set(x) for x in files.values()))

different = [
    p for p in sorted(common)
    if files["phase2-code"][p] != files["phase2-code-single-seed"][p]
]

print("\nCommon Python files:", len(common))
print("Different Python files:", different)

for name in folders:
    other = next(n for n in folders if n != name)
    unique = sorted(set(files[name]) - set(files[other]))
    print(f"Only in {name}:", unique[:30])

In [ ]:
import sys
from pathlib import Path

CODE_ROOT = Path("/content/drive/MyDrive/SVR_NEW/phase2-code")

sys.path.insert(0, str(CODE_ROOT))

from infra.seeds import set_global_seed
from infra.predictions_file import read_predictions_file, write_predictions_file

print("✅ Project imports successful")
print("Code root:", CODE_ROOT)

In [ ]:
required = [
    "torch",
    "DEVICE",
    "TRAIN_DATASET",
    "TASKS",
    "CLASSES_BY_TASK",
    "load_pretrained_model",
    "build_optimizer",
    "collate_batch",
    "task_losses",
    "kendall_total_loss",
    "autocast_context",
]

missing = [name for name in required if name not in globals()]

print("Missing variables/functions:", missing)

if not missing:
    ORIGINAL_AMP_ENABLED = globals().get("AMP_ENABLED", False)
    AMP_ENABLED = False

    print("✅ Ready for FP32 smoke test")
    print("AMP_ENABLED:", AMP_ENABLED)
else:
    print("⚠️ Run the notebook setup cells before the smoke test.")

In [ ]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working directory:", os.getcwd())

print("\n=== Runtime ===")
print("Variables loaded:", len(globals()))

print("\n=== Google Drive ===")
drive = Path("/content/drive/MyDrive")
print("Drive mounted:", drive.exists())

print("\n=== Multi-Task files ===")
for folder in [
    "/content/drive/MyDrive/SVR_NEW",
    "/content/drive/MyDrive/MARBERTv2_LR3e5_5Seeds",
    "/content/SVR_MARBERTv2_WORK",
    "/content/SVR_MARBERTv2_FINALIZE_WORK",
]:
    path = Path(folder)
    print(f"{path.name}: {'EXISTS' if path.exists() else 'MISSING'}")

print("\n=== GPU ===")
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch error:", str(e))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

print("Drive mounted:", DRIVE.exists())
print("SVR_NEW exists:", (DRIVE / "SVR_NEW").exists())
print(
    "Single-Task handoff exists:",
    (DRIVE / "MARBERTv2_LR2e5_COMPLETE" / "AYA_HANDOFF").exists()
)

In [ ]:
from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/SVR_NEW")

print("=== AVAILABLE NOTEBOOKS ===")

notebooks = sorted(ROOT.rglob("*.ipynb"))

for p in notebooks[:30]:
    print(p.relative_to(ROOT))

print("\nTotal notebooks:", len(notebooks))

print("\n=== REQUIRED SETUP STATUS ===")

required = [
    "torch", "DEVICE", "TRAIN_DATASET", "TASKS",
    "CLASSES_BY_TASK", "load_pretrained_model",
    "build_optimizer", "collate_batch", "task_losses",
    "kendall_total_loss", "autocast_context",
    "SHARED", "MODEL_NAME", "MODEL_REVISION",
    "AMP_ENABLED", "AMP_DTYPE", "GRADIENT_CLIP_NORM",
    "DataLoader",
]

missing = [name for name in required if name not in globals()]

print("Missing:", missing)

In [ ]:
from pathlib import Path
import json

ROOT = Path("/content/drive/MyDrive/SVR_NEW")

terms = [
    "kendall_total_loss",
    "SMOKE_MODEL",
    "log_variances",
]

matches = []

for path in ROOT.rglob("*"):
    if not path.is_file() or path.suffix not in (".py", ".ipynb"):
        continue

    try:
        content = path.read_text(encoding="utf-8")
    except (UnicodeError, OSError):
        continue

    found = [term for term in terms if term in content]

    if found:
        matches.append((path, found))

print("=== MULTI-TASK CODE SEARCH ===")

for path, found in matches[:40]:
    print("\nFILE:", path.relative_to(ROOT))
    print("MATCHES:", ", ".join(found))

print("\nTOTAL MATCHING FILES:", len(matches))

In [ ]:
nonfinite = []
finite_nonzero = 0

for name, param in SMOKE_MODEL.named_parameters():
    if param.grad is None:
        continue

    grad = param.grad.detach()

    if not torch.isfinite(grad).all():
        nonfinite.append(name)
    elif torch.count_nonzero(grad).item() > 0:
        finite_nonzero += 1

print("Parameters with non-finite gradients:", len(nonfinite))
print("Examples:", nonfinite[:10])
print("Parameters with finite non-zero gradients:", finite_nonzero)
print("GradScaler current scale:", smoke_scaler.get_scale())

## 9. Five-seed training loop, explicit opt-in

This is the five-seed confirmation at the already selected 3e-5 learning rate. Checkpoints are keyed by run ID and seed. Resume requires identical config, frozen-input hashes, and runtime identity. Each best model and tokenizer is saved per seed; `latest.pt` stores only the resumable training state. `RUN_FIVE_SEED_TRAINING` is false by default.

In [ ]:
import inspect
from infra.harness import evaluate_predictions

print(inspect.getsource(evaluate_predictions))

In [ ]:
import os
import random
import tempfile
from datetime import datetime, timezone

from safetensors.torch import load_file, save_file
from transformers import get_linear_schedule_with_warmup

RUN_ID = 'marbertv2-multitask-kendall-lr2e5-confirmation-v1'
SEEDS = [42, 1337, 2023, 7, 99]
if SHARED['experiment_seeds'] != SEEDS:
    raise ValueError(f'Frozen experiment seeds differ from the required order: {SHARED["experiment_seeds"]}')
if LEARNING_RATE != 2e-5 or EFFECTIVE_BATCH_SIZE != 16 or EPOCHS != 3:
    raise ValueError('This confirmation requires learning rate 2e-5, effective batch 16, and 3 epochs')
RUN_FIVE_SEED_TRAINING = True
RESUME = False
RUNS_ROOT = OUTPUT_ROOT / 'runs'
RUN_DIR = RUNS_ROOT / f'exp_{RUN_ID}'
RUN_DIRECTORY_CREATED_DURING_EXECUTION = False
EXCLUSIONS = {'customer_state': {'labels': ['unclear'],
    'reason': 'unclear has no train rows and is excluded from customer_state scoring only'}}

from infra.experiment_config import ExperimentConfig
from infra.harness import evaluate_predictions
from infra.metrics import seed_metrics
from infra.predictions_file import (predictions_path, read_predictions_file,
                                    write_predictions_file)
from infra.run_record import (build_run_record, load_run_record, make_seed_result,
                              save_run_record)
from infra.seeds import summarize_metrics

EXPERIMENT_CONFIG = ExperimentConfig(
    model=MODEL_NAME,
    dataset_version=SHARED['dataset_version'],
    learning_rate=LEARNING_RATE,
    batch_size=EFFECTIVE_BATCH_SIZE,
    epochs=EPOCHS,
    other_hyperparameters={
        'model_revision': MODEL_REVISION,
        'max_seq_len': MAX_LENGTH,
        'micro_batch_size': MICRO_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'experiment_seeds': list(SHARED['experiment_seeds']),
        'split_seed': SHARED['split_seed'],
        'architecture': 'shared_encoder_four_heads',
        'loss': 'weighted_cross_entropy_kendall_log_variance',
        'initial_log_variance': 0.0,
        'warmup_ratio': WARMUP_RATIO,
        'weight_decay': WEIGHT_DECAY,
        'dropout': DROPOUT,
        'gradient_clip_norm': GRADIENT_CLIP_NORM,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'precision': str(AMP_DTYPE) if AMP_ENABLED else 'fp32',
        'training_device': RUNTIME_DEVICE,
        'churn_weight_cap': CHURN_WEIGHT_CAP,
        'input_column': 'text_norm',
        'tasks': list(TASKS),
        'train_sha256': INPUT_SHA256['train'],
        'validation_sha256': INPUT_SHA256['validation'],
        'class_weights_sha256': INPUT_SHA256['class_weights'],
        'selection': 'highest_unweighted_mean_validation_macro_f1_then_earliest_epoch',
    },
)
RESUME_SIGNATURE = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'transformers': __import__('transformers').__version__, 'cuda': torch.version.cuda,
    'device': str(DEVICE), 'device_name': RUNTIME_DEVICE,
    'device_capability': list(torch.cuda.get_device_capability()) if torch.cuda.is_available() else None,
    'cudnn': torch.backends.cudnn.version() if torch.cuda.is_available() else None,
}
CONFIG_FINGERPRINT = hashlib.sha256(
    json.dumps({'experiment': EXPERIMENT_CONFIG.to_dict(), 'runtime': RESUME_SIGNATURE},
               sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()

def atomic_torch_save(value, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    handle, temporary_name = tempfile.mkstemp(prefix=f'.{destination.name}.', dir=destination.parent)
    os.close(handle)
    temporary = Path(temporary_name)
    try:
        torch.save(value, temporary)
        os.replace(temporary, destination)
    finally:
        if temporary.exists():
            temporary.unlink()

def atomic_safetensors_save(state, destination, metadata):
    destination.parent.mkdir(parents=True, exist_ok=True)
    handle, temporary_name = tempfile.mkstemp(prefix=f'.{destination.stem}.', suffix='.safetensors',
                                               dir=destination.parent)
    os.close(handle)
    temporary = Path(temporary_name)
    try:
        tensors = {key: value.detach().cpu().contiguous() for key, value in state.items()}
        save_file(tensors, str(temporary), metadata={key: str(value) for key, value in metadata.items()})
        os.replace(temporary, destination)
    finally:
        if temporary.exists():
            temporary.unlink()

def atomic_json_save(value, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    handle, temporary_name = tempfile.mkstemp(prefix=f'.{destination.name}.', dir=destination.parent,
                                               text=True)
    temporary = Path(temporary_name)
    try:
        with os.fdopen(handle, 'w', encoding='utf-8') as stream:
            json.dump(value, stream, indent=2, allow_nan=False)
            stream.write('\n')
        os.replace(temporary, destination)
    finally:
        if temporary.exists():
            temporary.unlink()

@torch.inference_mode()
def predict_validation(model):
    loader = DataLoader(VALIDATION_DATASET, batch_size=MICRO_BATCH_SIZE, shuffle=False,
                        collate_fn=collate_batch)
    model.eval()
    probabilities = {task: [] for task in TASKS}
    for model_inputs, _ in loader:
        model_inputs = {key: value.to(DEVICE) for key, value in model_inputs.items()}
        with autocast_context():
            logits = model(**model_inputs)
        for task in TASKS:
            probs = logits[task].float().softmax(dim=-1).cpu().tolist()
            if not np.isfinite(probs).all():
                raise FloatingPointError(f'Non-finite validation probabilities for {task}')
            probabilities[task].extend(probs)
    pred_rows = []
    for index, audio_file in enumerate(VALIDATION_DATASET.audio_files):
        pred_rows.append({'audio_file': audio_file, **{
            task: CLASSES_BY_TASK[task][int(np.argmax(probabilities[task][index]))]
            for task in TASKS
        }})
    report = evaluate_predictions('val', INPUTS.truth, pred_rows, CLASSES_BY_TASK, EXCLUSIONS)
    return probabilities, report

def make_loader(seed):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(TRAIN_DATASET, batch_size=MICRO_BATCH_SIZE, shuffle=True, generator=generator,
                      collate_fn=collate_batch), generator

def train_one_seed(seed):
    if seed not in SEEDS:
        raise ValueError(f'Unexpected seed {seed}; expected exactly {SEEDS}')
    seed_dir = RUN_DIR / 'checkpoints' / f'seed_{seed}'
    latest_path = seed_dir / 'latest.pt'
    best_path = seed_dir / 'model.safetensors'
    architecture_path = seed_dir / 'architecture.json'
    tokenizer_dir = seed_dir / 'tokenizer'
    result_path = seed_dir / 'validation_result.json'
    global RUN_DIRECTORY_CREATED_DURING_EXECUTION
    if RUN_DIR.exists() and not RESUME and not RUN_DIRECTORY_CREATED_DURING_EXECUTION:
        raise FileExistsError(f'Run directory already exists; choose a fresh RUN_ID or explicitly resume: {RUN_DIR}')
    run_record_path = RUN_DIR / 'run_record.json'
    if run_record_path.exists():
        raise FileExistsError(f'Run is already finalized and cannot be trained or overwritten: {run_record_path}')
    if RUN_DIR.exists() and RESUME:
        config_path = RUN_DIR / 'config.json'
        if not config_path.is_file():
            raise FileExistsError(f'Cannot resume without the original run config: {RUN_DIR}')
        existing_config = json.loads(config_path.read_text(encoding='utf-8'))
        if (existing_config.get('run_id') != RUN_ID
                or existing_config.get('config_fingerprint') != CONFIG_FINGERPRINT
                or existing_config.get('runtime_signature') != RESUME_SIGNATURE
                or existing_config.get('input_sha256') != INPUT_SHA256):
            raise ValueError('Existing output directory belongs to another run or configuration')
    if result_path.exists():
        if not RESUME:
            raise FileExistsError(f'{result_path} exists; use RESUME=True for this exact run')
        saved_result = json.loads(result_path.read_text(encoding='utf-8'))
        if (saved_result['run_id'] != RUN_ID or saved_result['seed'] != seed
                or saved_result['config_fingerprint'] != CONFIG_FINGERPRINT
                or saved_result.get('input_sha256') != INPUT_SHA256):
            raise ValueError(f'Completed result identity mismatch: {result_path}')
        required_seed_artifacts = (best_path, architecture_path, tokenizer_dir / 'tokenizer_config.json')
        missing_seed_artifacts = [path for path in required_seed_artifacts if not path.is_file()]
        if missing_seed_artifacts:
            raise FileNotFoundError(f'Completed seed is missing handoff artifacts: {missing_seed_artifacts}')
        architecture = json.loads(architecture_path.read_text(encoding='utf-8'))
        if architecture.get('config_fingerprint') != CONFIG_FINGERPRINT or architecture.get('seed') != seed:
            raise ValueError(f'Architecture metadata identity mismatch: {architecture_path}')
        print(f'Seed {seed} already completed, keeping its verified result.')
        return saved_result
    if latest_path.exists() and not RESUME:
        raise FileExistsError(f'{latest_path} exists; set RESUME=True to continue this seed')
    if not RUN_DIR.exists():
        RUN_DIR.mkdir(parents=True, exist_ok=False)
        RUN_DIRECTORY_CREATED_DURING_EXECUTION = True
        atomic_json_save({'run_id': RUN_ID, 'config': EXPERIMENT_CONFIG.to_dict(),
                          'config_fingerprint': CONFIG_FINGERPRINT, 'runtime_signature': RESUME_SIGNATURE,
                          'input_sha256': INPUT_SHA256, 'created_utc': datetime.now(timezone.utc).isoformat()},
                         RUN_DIR / 'config.json')
    else:
        config_path = RUN_DIR / 'config.json'
        if not config_path.is_file():
            raise FileExistsError(f'Output directory exists without a run config: {RUN_DIR}')
        existing_config = json.loads(config_path.read_text(encoding='utf-8'))
        if (existing_config['run_id'] != RUN_ID
                or existing_config['config_fingerprint'] != CONFIG_FINGERPRINT
                or existing_config.get('runtime_signature') != RESUME_SIGNATURE
                or existing_config.get('input_sha256') != INPUT_SHA256):
            raise ValueError('Existing output directory belongs to another run or configuration')
    if not tokenizer_dir.exists():
        tokenizer.save_pretrained(tokenizer_dir)
    elif not (tokenizer_dir / 'tokenizer_config.json').is_file():
        raise FileExistsError(f'Existing tokenizer directory is incomplete: {tokenizer_dir}')
    set_global_seed(seed)
    model = load_pretrained_model().to(DEVICE)
    loader, shuffle_generator = make_loader(seed)
    optimizer = build_optimizer(model)
    updates_per_epoch = math.ceil(len(loader) / GRADIENT_ACCUMULATION_STEPS)
    total_updates = updates_per_epoch * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, math.ceil(total_updates * WARMUP_RATIO), total_updates)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED and AMP_DTYPE == torch.float16)
    start_epoch, best_score, best_epoch, stale_epochs, history = 0, -1.0, None, 0, []
    if latest_path.exists():
        state = torch.load(latest_path, map_location='cpu', weights_only=False)
        expected = {'run_id': RUN_ID, 'seed': seed, 'config_fingerprint': CONFIG_FINGERPRINT,
                    'input_sha256': INPUT_SHA256, 'runtime_signature': RESUME_SIGNATURE}
        if any(state.get(key) != value for key, value in expected.items()):
            raise ValueError(f'Checkpoint identity or frozen input mismatch: {latest_path}')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        start_epoch, best_score, best_epoch = state['epoch'], state['best_score'], state['best_epoch']
        stale_epochs, history = state['stale_epochs'], state['history']
        shuffle_generator.set_state(state['shuffle_rng'])
        random.setstate(state['python_rng'])
        np.random.set_state(state['numpy_rng'])
        torch.set_rng_state(state['torch_rng'])
        if torch.cuda.is_available() and state['cuda_rng'] is not None:
            torch.cuda.set_rng_state_all(state['cuda_rng'])
        print(f'Resuming seed {seed} after epoch {start_epoch}')
    seed_dir.mkdir(parents=True, exist_ok=True)
    for epoch in range(start_epoch + 1, EPOCHS + 1):
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            break
        model.train()
        optimizer.zero_grad(set_to_none=True)
        batch_losses, successful_updates = [], 0
        for batch_index, (model_inputs, labels) in enumerate(loader):
            model_inputs = {key: value.to(DEVICE) for key, value in model_inputs.items()}
            labels = {key: value.to(DEVICE) for key, value in labels.items()}
            with autocast_context():
                logits = model(**model_inputs)
                losses = task_losses(logits, labels)
                total_loss = kendall_total_loss(model, losses)
            if not torch.isfinite(total_loss):
                raise FloatingPointError(f'Non-finite training loss, seed {seed}, epoch {epoch}')
            scaler.scale(total_loss / GRADIENT_ACCUMULATION_STEPS).backward()
            batch_losses.append(float(total_loss.detach()))
            should_step = ((batch_index + 1) % GRADIENT_ACCUMULATION_STEPS == 0
                           or batch_index + 1 == len(loader))
            if should_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= previous_scale:
                    scheduler.step()
                    successful_updates += 1
                optimizer.zero_grad(set_to_none=True)
        if successful_updates == 0:
            raise RuntimeError(f'No optimizer updates completed in seed {seed}, epoch {epoch}')
        probabilities, report = predict_validation(model)
        selection_score = float(report['combined_macro_f1']['mean'])
        improved = selection_score > best_score
        if improved:
            best_score, best_epoch, stale_epochs = selection_score, epoch, 0
            atomic_safetensors_save(model.state_dict(), best_path,
                                    {'run_id': RUN_ID, 'seed': seed, 'epoch': epoch,
                                     'config_fingerprint': CONFIG_FINGERPRINT,
                                     'model': MODEL_NAME, 'revision': MODEL_REVISION})
            architecture = {
                'run_id': RUN_ID, 'seed': seed, 'epoch': epoch,
                'config_fingerprint': CONFIG_FINGERPRINT,
                'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION,
                'architecture': 'shared_encoder_four_heads',
                'encoder_hidden_size': int(model.encoder.config.hidden_size),
                'encoder_config': model.encoder.config.to_dict(),
                'head_dimensions': {task: len(CLASSES_BY_TASK[task]) for task in TASKS},
                'dropout': DROPOUT, 'max_sequence_length': MAX_LENGTH,
                'tasks': list(TASKS),
                'label_mappings': {task: {label: index for index, label in enumerate(CLASSES_BY_TASK[task])}
                                   for task in TASKS},
                'class_weights': WEIGHTS,
                'loss_class_weights': {task: CLASS_WEIGHT_TENSORS[task].detach().cpu().tolist()
                                       for index, task in enumerate(TASKS)},
                'churn_weight_cap': CHURN_WEIGHT_CAP,
                'uncertainty_parameter': 'log_variances',
                'uncertainty_parameters': model.log_variances.detach().cpu().tolist(),
                'validation_mean_macro_f1': selection_score,
            }
            atomic_json_save(architecture, architecture_path)
        else:
            stale_epochs += 1
        history.append({'epoch': epoch, 'mean_batch_loss': float(np.mean(batch_losses)),
                        'validation_mean_macro_f1': selection_score, 'best_epoch': best_epoch,
                        'optimizer_updates': successful_updates,
                        'log_variances': model.log_variances.detach().cpu().tolist()})
        atomic_torch_save({
            'run_id': RUN_ID, 'seed': seed, 'config_fingerprint': CONFIG_FINGERPRINT,
            'runtime_signature': RESUME_SIGNATURE,
            'input_sha256': INPUT_SHA256, 'epoch': epoch, 'best_score': best_score,
            'best_epoch': best_epoch, 'stale_epochs': stale_epochs, 'history': history,
            'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
            'shuffle_rng': shuffle_generator.get_state(), 'python_rng': random.getstate(),
            'numpy_rng': np.random.get_state(), 'torch_rng': torch.get_rng_state(),
            'cuda_rng': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }, latest_path)
        print(f'seed {seed} epoch {epoch}: mean validation macro-F1={selection_score:.4f}; best={best_score:.4f}')
    if best_epoch is None or not best_path.is_file() or not architecture_path.is_file():
        raise RuntimeError(f'No validation-selected checkpoint exists for seed {seed}')
    architecture = json.loads(architecture_path.read_text(encoding='utf-8'))
    if (architecture.get('run_id') != RUN_ID or architecture.get('seed') != seed
            or architecture.get('config_fingerprint') != CONFIG_FINGERPRINT
            or architecture.get('model_revision') != MODEL_REVISION):
        raise ValueError(f'Selected checkpoint identity mismatch: {architecture_path}')
    model.load_state_dict(load_file(str(best_path), device='cpu'))
    probabilities, report = predict_validation(model)
    result = {'run_id': RUN_ID, 'seed': seed, 'config_fingerprint': CONFIG_FINGERPRINT,
              'input_sha256': INPUT_SHA256, 'best_epoch': architecture['epoch'],
              'history': history, 'report': report, 'probabilities': probabilities,
              'log_variances': model.log_variances.detach().cpu().tolist()}
    atomic_json_save(result, result_path)
    del model, optimizer, scheduler, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

if RUN_FIVE_SEED_TRAINING:
    SEED_RESULTS = [train_one_seed(seed) for seed in SEEDS]
else:
    print('Five-seed training is disabled. Set RUN_FIVE_SEED_TRAINING = True only after approval.')

In [ ]:
from pathlib import Path
import json
import csv

# Separate from Single-Task inference handoff
results_dir = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5/RESULTS"
)
results_dir.mkdir(parents=True, exist_ok=True)

results = [
    {"seed": 42, "epoch1": 0.6692, "epoch2": 0.7422, "epoch3": 0.7427, "best_epoch": 3, "best_f1": 0.7427},
    {"seed": 1337, "epoch1": 0.6625, "epoch2": 0.7618, "epoch3": 0.7560, "best_epoch": 2, "best_f1": 0.7618},
    {"seed": 2023, "epoch1": 0.6655, "epoch2": 0.7183, "epoch3": 0.7453, "best_epoch": 3, "best_f1": 0.7453},
    {"seed": 7, "epoch1": 0.6703, "epoch2": 0.7271, "epoch3": 0.7558, "best_epoch": 3, "best_f1": 0.7558},
    {"seed": 99, "epoch1": 0.6787, "epoch2": 0.7532, "epoch3": 0.7712, "best_epoch": 3, "best_f1": 0.7712},
]

report = {
    "experiment": "MARBERTv2_MultiTask_Kendall_LR2e5",
    "model": "UBC-NLP/MARBERTv2",
    "learning_rate": 2e-5,
    "metric": "mean_validation_macro_f1",
    "mean_best_validation_macro_f1": sum(
        r["best_f1"] for r in results
    ) / len(results),
    "results": results,
    "note": "Validation results from supplied training logs; not test scores."
}

json_path = results_dir / "validation_results.json"
csv_path = results_dir / "validation_results.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

# Verify files can be read back
with open(json_path, encoding="utf-8") as f:
    saved_report = json.load(f)

assert len(saved_report["results"]) == 5
assert csv_path.is_file()

print("Saved successfully to Google Drive!")
print("Folder:", results_dir)
print("Mean best validation macro-F1:",
      round(saved_report["mean_best_validation_macro_f1"], 5))
print("Files:", json_path.name, csv_path.name)

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

RUN_ROOT = Path(
    "/content/drive/MyDrive/SVR_NEW/"
    "marbertv2-multitask-output/runs/"
    "exp_marbertv2-multitask-kendall-lr3e5-confirmation-v1"
)

checkpoint_path = RUN_ROOT / "checkpoints" / "seed_1337" / "latest.pt"

print("Checkpoint exists:", checkpoint_path.is_file())

if checkpoint_path.is_file():
    print("Size (GiB):", round(checkpoint_path.stat().st_size / 1024**3, 2))

In [ ]:
import torch

unsafe_globals = torch.serialization.get_unsafe_globals_in_checkpoint(
    checkpoint_path
)

print("Globals requiring review:")
for item in unsafe_globals:
    print("-", item)

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5"
)

assert ROOT.is_dir(), f"Folder not found: {ROOT}"

checkpoint_files = sorted(
    p for p in ROOT.rglob("*.pt")
    if "checkpoint" in p.name.lower()
    or p.name.lower() in {"best.pt", "latest.pt"}
)

print("Total checkpoint files:", len(checkpoint_files))

for i, path in enumerate(checkpoint_files, 1):
    print(f"\n{i}. {path.relative_to(ROOT)}")
    print(f"   Size: {path.stat().st_size / (1024**2):.1f} MB")

In [ ]:
import torch
import numpy as np
import pandas as pd

from torch.torch_version import TorchVersion
from torch._subclasses.fake_tensor import FakeTensorMode
from numpy._core.multiarray import _reconstruct

allowed = [
    TorchVersion,
    np.dtype,
    np.ndarray,
    _reconstruct,
    type(np.dtype(np.uint32)),
]

expected_run = "exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"

expected_scores = {
    42: (3, 0.7427),
    1337: (2, 0.7618),
    2023: (3, 0.7453),
    7: (3, 0.7558),
    99: (3, 0.7712),
}

rows = []

for path in checkpoint_files:
    print(f"Inspecting: {path.parent.name}")

    with torch.serialization.safe_globals(allowed):
        with FakeTensorMode():
            ckpt = torch.load(
                path,
                map_location="cpu",
                weights_only=True
            )

    seed = int(path.parent.name.replace("seed_", ""))
    expected_epoch, expected_f1 = expected_scores[seed]

    run_id = ckpt.get("run_id")
    best_epoch = ckpt.get("best_epoch")
    best_score = ckpt.get("best_score")

    rows.append({
        "seed": seed,
        "checkpoint_epoch": ckpt.get("epoch"),
        "best_epoch": best_epoch,
        "expected_best_epoch": expected_epoch,
        "best_score": best_score,
        "expected_best_score": expected_f1,
        "run_id": run_id,
        "run_id_match": run_id == expected_run,
        "epoch_match": best_epoch == expected_epoch,
        "score_match": (
            abs(float(best_score) - expected_f1) < 0.0001
            if best_score is not None else False
        ),
        "has_model": isinstance(ckpt.get("model"), dict),
    })

    del ckpt

df_audit = pd.DataFrame(rows).sort_values("seed")

display(df_audit)

print("\nRun IDs:")
print(df_audit["run_id"].value_counts().to_string())

print("\nValidation checks:")
print("Run ID matches:", int(df_audit["run_id_match"].sum()), "/ 5")
print("Best epochs match:", int(df_audit["epoch_match"].sum()), "/ 5")
print("Best scores match:", int(df_audit["score_match"].sum()), "/ 5")
print("Model state exists:", int(df_audit["has_model"].sum()), "/ 5")

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5"
)

run_dir = ROOT / "runs" / (
    "exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

extensions = {".pt", ".pth", ".bin", ".safetensors"}

files = sorted(
    p for p in run_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
)

print("Total model/checkpoint files:", len(files))

for path in files:
    print(path.relative_to(run_dir))

In [ ]:
from safetensors import safe_open
from pathlib import Path
import torch
import pandas as pd

rows = []

for seed in [42, 1337, 2023, 7, 99]:
    seed_dir = run_dir / "checkpoints" / f"seed_{seed}"
    model_path = seed_dir / "model.safetensors"
    latest_path = seed_dir / "latest.pt"

    with safe_open(str(model_path), framework="pt", device="cpu") as f:
        keys = list(f.keys())
        metadata = f.metadata() or {}

    rows.append({
        "seed": seed,
        "model_exists": model_path.is_file(),
        "model_size_mb": round(model_path.stat().st_size / 1024**2, 2),
        "tensor_count": len(keys),
        "metadata": metadata,
        "latest_exists": latest_path.is_file()
    })

df_models = pd.DataFrame(rows)

display(df_models)

In [ ]:
from safetensors import safe_open
import pandas as pd

expected = {
    42: (3, 0.7427),
    1337: (2, 0.7618),
    2023: (3, 0.7453),
    7: (3, 0.7558),
    99: (3, 0.7712),
}

rows = []

for seed, (expected_epoch, expected_score) in expected.items():
    path = (
        run_dir / "checkpoints"
        / f"seed_{seed}"
        / "model.safetensors"
    )

    with safe_open(str(path), framework="pt", device="cpu") as f:
        metadata = f.metadata() or {}

    saved_epoch = metadata.get("epoch")
    saved_score = metadata.get("best_score")

    rows.append({
        "seed": seed,
        "saved_epoch": saved_epoch,
        "expected_best_epoch": expected_epoch,
        "epoch_match": (
            str(saved_epoch) == str(expected_epoch)
            if saved_epoch is not None else None
        ),
        "saved_score": saved_score,
        "expected_best_score": expected_score,
        "run_id": metadata.get("run_id"),
        "metadata": metadata
    })

df_best = pd.DataFrame(rows)

for _, row in df_best.iterrows():
    print(f"\n{'='*50}")
    print(f"SEED {row['seed']}")
    print(f"{'='*50}")
    print("Full metadata:", row["metadata"])
    print("Epoch matches:", row["epoch_match"])

In [ ]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

results_dir = run_dir / "RESULTS"
results_dir.mkdir(parents=True, exist_ok=True)

validation = {
    42:   [0.6692, 0.7422, 0.7427],
    1337: [0.6625, 0.7618, 0.7560],
    2023: [0.6655, 0.7183, 0.7453],
    7:    [0.6703, 0.7271, 0.7558],
    99:   [0.6787, 0.7532, 0.7712],
}

run_id = "marbertv2-multitask-kendall-lr2e5-confirmation-v1"

rows = []

for seed, scores in validation.items():
    model_path = (
        run_dir / "checkpoints" /
        f"seed_{seed}" / "model.safetensors"
    )

    with safe_open(str(model_path), framework="pt", device="cpu") as f:
        metadata = f.metadata() or {}
        tensor_count = len(f.keys())

    best_epoch = int(max(range(3), key=lambda i: scores[i])) + 1

    rows.append({
        "seed": seed,
        "epoch_1_f1": scores[0],
        "epoch_2_f1": scores[1],
        "epoch_3_f1": scores[2],
        "best_epoch": best_epoch,
        "best_validation_f1": max(scores),
        "saved_model_epoch": int(metadata["epoch"]),
        "epoch_match": int(metadata["epoch"]) == best_epoch,
        "run_id_match": metadata.get("run_id") == run_id,
        "tensor_count": tensor_count,
        "config_fingerprint": metadata.get("config_fingerprint"),
        "model_revision": metadata.get("revision"),
    })

df = pd.DataFrame(rows).sort_values("seed")

assert df["epoch_match"].all()
assert df["run_id_match"].all()

csv_path = results_dir / "validation_checkpoint_audit.csv"
json_path = results_dir / "validation_checkpoint_audit.json"

df.to_csv(csv_path, index=False)

report = {
    "run_id": run_id,
    "model": "UBC-NLP/MARBERTv2",
    "learning_rate": 2e-5,
    "metric": "mean_validation_macro_f1",
    "mean_best_validation_f1": float(df["best_validation_f1"].mean()),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "validation_source": "User-provided training logs (rounded to 4 decimals)",
    "verification": "Checkpoint metadata and file structure; no inference smoke test",
    "results": df.to_dict(orient="records"),
}

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

# Verify saved files
assert csv_path.is_file()
assert json_path.is_file()

with open(json_path, encoding="utf-8") as f:
    verified = json.load(f)

assert len(verified["results"]) == 5

print("SUCCESS: Reports saved to Google Drive")
print("Location:", results_dir)
print("Mean Best Validation F1:",
      round(verified["mean_best_validation_f1"], 5))
print("Files:", csv_path.name, json_path.name)

In [ ]:
from pathlib import Path

results_dir = run_dir / "RESULTS"

for filename in [
    "validation_checkpoint_audit.csv",
    "validation_checkpoint_audit.json"
]:
    path = results_dir / filename
    print(f"{filename}: {'SAVED' if path.is_file() else 'MISSING'}")

In [ ]:
from pathlib import Path
import pandas as pd

RUN_NAME = "exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"

# Google Drive experiment directory
drive_run = (
    Path("/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5")
    / "runs"
    / RUN_NAME
)

print("DRIVE RUN EXISTS:", drive_run.is_dir())

# Inventory function
def inventory(root):
    if not root.is_dir():
        return pd.DataFrame(columns=["file", "size_mb"])

    return pd.DataFrame([
        {
            "file": str(p.relative_to(root)),
            "size_mb": round(p.stat().st_size / 1024**2, 3)
        }
        for p in root.rglob("*")
        if p.is_file()
    ])

# Google Drive inventory
drive_files = inventory(drive_run)

print("\n=== GOOGLE DRIVE INVENTORY ===")
print("Total files:", len(drive_files))
print(
    "Total size (GB):",
    round(drive_files["size_mb"].sum() / 1024, 3)
)

display(
    drive_files.sort_values("file").reset_index(drop=True)
)

# Search for possible local training runs
print("\n=== POSSIBLE LOCAL TRAINING RUNS ===")

local_roots = [
    Path("/content"),
    Path("/content/outputs"),
    Path("/content/runs"),
]

matches = set()

for root in local_roots:
    candidate = root / RUN_NAME
    if candidate.is_dir():
        matches.add(str(candidate))

    candidate = root / "runs" / RUN_NAME
    if candidate.is_dir():
        matches.add(str(candidate))

for path in sorted(matches):
    print(path)

if not matches:
    print(
        "No matching local run found in the checked locations."
    )

print("\n=== INVENTORY COMPLETE ===")

In [ ]:
from pathlib import Path

root = Path("/content")
drive_mount = Path("/content/drive")

local_files = []

for p in root.rglob("*"):
    if not p.is_file():
        continue

    if p.is_relative_to(drive_mount):
        continue

    local_files.append({
        "path": str(p),
        "size_mb": round(p.stat().st_size / (1024**2), 3)
    })

print("LOCAL FILES OUTSIDE DRIVE:", len(local_files))

for item in sorted(local_files, key=lambda x: -x["size_mb"])[:60]:
    print(f"{item['size_mb']:>10.3f} MB | {item['path']}")

In [ ]:
from pathlib import Path
import shutil
import hashlib
import pandas as pd

SOURCE = Path("/content")

DEST = Path(
    "/content/drive/MyDrive/"
    "MARBERTv2_MULTITASK_LR2e5/"
    "COLAB_LOCAL_BACKUP"
)

DEST.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

results = []

# Collect local files, excluding mounted Drive
files = [
    p for p in SOURCE.rglob("*")
    if p.is_file()
    and not p.is_relative_to(SOURCE / "drive")
]

print("FILES TO BACK UP:", len(files))

for src in files:
    relative = src.relative_to(SOURCE)
    dst = DEST / relative

    try:
        dst.parent.mkdir(parents=True, exist_ok=True)

        shutil.copy2(src, dst)

        source_hash = sha256(src)
        backup_hash = sha256(dst)

        verified = source_hash == backup_hash

        results.append({
            "file": str(relative),
            "status": "VERIFIED" if verified else "MISMATCH",
            "size_bytes": src.stat().st_size,
            "sha256": source_hash
        })

        print(
            "VERIFIED" if verified else "MISMATCH",
            "|",
            relative
        )

    except Exception as e:
        results.append({
            "file": str(relative),
            "status": "ERROR",
            "size_bytes": None,
            "sha256": None
        })
        print("ERROR |", relative, "|", str(e))

df = pd.DataFrame(results)

# Save verification manifest
manifest = DEST / "backup_manifest.csv"
df.to_csv(manifest, index=False)

print("\n=== BACKUP SUMMARY ===")
print("Total files:", len(df))
print("Verified:", (df.status == "VERIFIED").sum())
print("Failed:", (df.status != "VERIFIED").sum())
print("Backup location:", DEST)

if (df.status == "VERIFIED").all() and len(df) > 0:
    print("\nSUCCESS: ALL LOCAL FILES BACKED UP AND VERIFIED")
else:
    print("\nWARNING: SOME FILES NEED REVIEW")

In [ ]:
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5/"
    "runs/exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1/"
    "checkpoints/seed_42/latest.pt"
)

print("Checkpoint:", checkpoint_path)
print("Exists:", checkpoint_path.is_file())

In [ ]:
import torch
import numpy as np

from torch.torch_version import TorchVersion
from torch._subclasses.fake_tensor import FakeTensorMode
from numpy._core.multiarray import _reconstruct

allowed = [
    TorchVersion,
    np.dtype,
    np.ndarray,
    _reconstruct,
    type(np.dtype(np.uint32)),
]

with torch.serialization.safe_globals(allowed):
    with FakeTensorMode():
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

history = checkpoint["history"]

print("History type:", type(history).__name__)
print("History length:", len(history))

if isinstance(history, list):
    for item in history:
        print(item)
elif isinstance(history, dict):
    print("History keys:", list(history.keys()))

del checkpoint

In [ ]:
from pathlib import Path
import torch
import numpy as np

from torch.torch_version import TorchVersion
from torch._subclasses.fake_tensor import FakeTensorMode
from numpy._core.multiarray import _reconstruct

# Correct checkpoint path: Multi-Task LR2e-5, Seed 42
checkpoint_path = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5/"
    "runs/exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1/"
    "checkpoints/seed_42/latest.pt"
)

assert checkpoint_path.is_file(), "Checkpoint file not found!"

# Allow required metadata types
allowed = [
    TorchVersion,
    np.dtype,
    np.ndarray,
    _reconstruct,
    type(np.dtype(np.uint32)),
]

# Read checkpoint metadata without loading model tensors
with torch.serialization.safe_globals(allowed):
    with FakeTensorMode():
        ckpt = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

print("=== CHECKPOINT METADATA ===")

for key in [
    "run_id",
    "seed",
    "config_fingerprint",
    "runtime_signature",
    "input_sha256",
]:
    print(f"\n{key}:")
    print(ckpt.get(key))

# Verify experiment identity
expected_run_id = (
    "marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

print("\n=== VERIFICATION ===")
print("Correct run:", ckpt.get("run_id") == expected_run_id)
print("Correct seed:", ckpt.get("seed") == 42)

del ckpt

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")
notebook_name = "marbertv2_multitask_colab.ipynb"

matches = list(drive.rglob(notebook_name))

print("NOTEBOOK FOUND:", len(matches))

for path in matches:
    print("Location:", path)
    print("Size (KB):", round(path.stat().st_size / 1024, 2))

In [ ]:
from safetensors import safe_open

for seed in [42, 1337]:
    path = RUN_ROOT / "checkpoints" / f"seed_{seed}" / "model.safetensors"

    print(f"\nChecking Seed {seed}...")

    try:
        count = 0

        with safe_open(str(path), framework="pt", device="cpu") as f:
            for key in f.keys():
                tensor = f.get_tensor(key)
                count += 1
                del tensor

        print(f"PASS: Read {count} tensors successfully")

    except Exception as e:
        print(f"FAIL: {type(e).__name__}: {e}")

In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/SVR_NEW")

print("Training notebooks and Python files:\n")

for path in sorted(project.rglob("*")):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {".py", ".ipynb"}:
        continue

    print(path.relative_to(project))

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/SVR_NEW")

terms = [
    "marbertv2-multitask-kendall",
    "validation_mean_macro_f1",
    "log_variances",
    "config_fingerprint",
]

print("Searching project files...\n")

for path in root.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {".py", ".ipynb", ".json", ".txt", ".md"}:
        continue

    if path.stat().st_size > 20 * 1024 * 1024:
        continue

    try:
        content = path.read_text(
            encoding="utf-8",
            errors="replace"
        ).lower()
    except OSError:
        continue

    matches = [term for term in terms if term in content]

    if matches:
        print(path.relative_to(root))
        print("Matches:", ", ".join(matches))
        print()

## 10. Per-task validation macro-F1 and confusion matrices

The final selected epoch for each seed is evaluated on validation. The existing harness enforces the full row match and the customer-state `unclear` exclusion. The confusion matrices are pooled counts across seeds, with true labels on rows and predictions on columns. I also report the existing validation-only churn threshold analysis as supplemental and potentially optimistic; the official comparison and prediction exports continue to use argmax.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

def load_seed_results():
    results = []
    for seed in SEEDS:
        path = RUN_DIR / 'checkpoints' / f'seed_{seed}' / 'validation_result.json'
        if not path.is_file():
            raise FileNotFoundError(f'Missing completed validation result for seed {seed}: {path}')
        result = json.loads(path.read_text(encoding='utf-8'))
        if (result['run_id'] != RUN_ID or result['seed'] != seed
                or result['config_fingerprint'] != CONFIG_FINGERPRINT
                or result.get('input_sha256') != INPUT_SHA256):
            raise ValueError(f'Result identity mismatch: {path}')
        results.append(result)
    return results

if RUN_DIR.is_dir() and all((RUN_DIR / 'checkpoints' / f'seed_{seed}' / 'validation_result.json').is_file()
                            for seed in SEEDS):
    SEED_RESULTS = load_seed_results()
    metric_rows = []
    for result in SEED_RESULTS:
        for task in TASKS:
            task_report = result['report']['fields'][task]
            metric_rows.append({'seed': result['seed'], 'task': task,
                                'macro_precision': task_report['macro']['precision'],
                                'macro_recall': task_report['macro']['recall'],
                                'macro_f1': task_report['macro']['f1'],
                                'accuracy': task_report['accuracy'],
                                'selected_epoch': result['best_epoch']})
    validation_metrics = pd.DataFrame(metric_rows)
    display(validation_metrics)
    from marbertv2_churn import churn_analysis
    churn_rows = []
    for result in SEED_RESULTS:
        churn_probabilities = {
            audio_file: {'churn_risk': {
                label: float(result['probabilities']['churn_risk'][index][class_index])
                for class_index, label in enumerate(CLASSES_BY_TASK['churn_risk'])
            }}
            for index, audio_file in enumerate(VALIDATION_DATASET.audio_files)
        }
        churn_report = churn_analysis(INPUTS.truth, churn_probabilities, CLASSES_BY_TASK['churn_risk'])
        churn_rows.append({'seed': result['seed'], 'validation_threshold': churn_report['threshold'],
                           'average_precision': churn_report['average_precision'],
                           'argmax_balanced_accuracy': churn_report['argmax_balanced_accuracy'],
                           'tuned_balanced_accuracy': churn_report['tuned_balanced_accuracy'],
                           'tuned_macro_f1': churn_report['tuned_macro_f1'],
                           'at_risk_true_positives': churn_report['true_positive'],
                           'at_risk_support': churn_report['at_risk_support']})
    display(pd.DataFrame(churn_rows))
    for task in TASKS:
        reports = [result['report']['fields'][task] for result in SEED_RESULTS]
        matrix = np.sum([np.asarray(report['confusion_matrix']['counts']) for report in reports], axis=0)
        labels = CLASSES_BY_TASK[task]
        fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.65), max(4, len(labels) * 0.55)))
        image = ax.imshow(matrix, cmap='Blues')
        ax.set(title=f'{task}, pooled across five seeds', xlabel='Predicted', ylabel='True')
        ax.set_xticks(range(len(labels)), labels, rotation=75, ha='right', fontsize=8)
        ax.set_yticks(range(len(labels)), labels, fontsize=8)
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        plt.show()
else:
    print('Validation report is pending. Complete the explicitly enabled training loop first.')

In [ ]:
from pathlib import Path

expected_run = "marbertv2-multitask-kendall-lr2e5-confirmation-v1"

assert RUN_ID == expected_run, "Wrong experiment RUN_ID!"
assert len(validation_metrics) == 20, "Unexpected metrics row count!"

results_dir = RUN_DIR / "RESULTS"
results_dir.mkdir(parents=True, exist_ok=True)

output = results_dir / "validation_per_task_per_seed.csv"

validation_metrics.to_csv(output, index=False)

print("SAVED:", output)
print("EXISTS:", output.is_file())
print("ROWS:", len(validation_metrics))

## 11. Mean and sample standard deviation across seeds

I report the mean and sample standard deviation (`ddof=1`) for macro precision, macro recall, macro-F1, and accuracy for each task. The unweighted mean of the four task macro-F1 values is shown alongside the task values.

In [ ]:
if 'validation_metrics' in globals():
    aggregate_rows = []
    for task in TASKS:
        subset = validation_metrics[validation_metrics['task'] == task]
        row = {'task': task}
        for metric in ('macro_precision', 'macro_recall', 'macro_f1', 'accuracy'):
            row[f'{metric}_mean'] = float(subset[metric].mean())
            row[f'{metric}_sample_sd'] = float(subset[metric].std(ddof=1))
        aggregate_rows.append(row)
    aggregate_table = pd.DataFrame(aggregate_rows)
    combined_mean = float(aggregate_table['macro_f1_mean'].mean())
    combined_sd_by_seed = float(pd.Series([
        result['report']['combined_macro_f1']['mean'] for result in SEED_RESULTS
    ]).std(ddof=1))
    display(aggregate_table)
    print(f'Unweighted mean of task mean macro-F1: {combined_mean:.4f}')
    print(f'Sample SD of the per-seed combined mean macro-F1: {combined_sd_by_seed:.4f}')
else:
    print('Five-seed aggregate is pending.')

## 12. Comparison with completed single-task and TF-IDF validation baselines

These are validation references only. Single-task metrics are read from an explicitly configured run record and are pending when no verified record is provided. TF-IDF values come from the existing Phase 2 report. Neither reference is a test result.

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

keywords = [
    "single",
    "run_record",
    "validation_result",
    "aggregate",
    "metrics",
]

matches = []

for path in drive.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {".json", ".csv"}:
        continue

    if any(word in path.name.lower() for word in keywords):
        matches.append(path)

print("FOUND:", len(matches))

for path in sorted(matches):
    print(path)

In [ ]:
from pathlib import Path
import json

single_task_path = Path(
    "/content/drive/MyDrive/MARBERTv2_LR2e5_COMPLETE/"
    "runs/exp_marbertv2-lr2e5-clean-20261119/"
    "run_record (1).json"
)

assert single_task_path.is_file(), "Single-task run record not found!"

record = json.loads(single_task_path.read_text(encoding="utf-8"))

print("=== SINGLE-TASK RUN RECORD ===")

print("Run ID:", record.get("run_id"))
print("Evaluated split:", record.get("evaluated_split"))
print("Dataset:", record.get("dataset"))
print("Model:", record.get("config", {}).get("model"))

print("\n=== SEEDS ===")
for entry in record.get("per_seed", []):
    print(
        "Seed:", entry.get("seed"),
        "| Run:", entry.get("run_id"),
        "| Split:", entry.get("split")
    )

print("\n=== AGGREGATE RESULTS ===")
for key, value in record.get("aggregate", {}).items():
    print(key, ":", value)

print("\n=== MODEL CONFIG ===")
print(record.get("config"))

In [ ]:
from pathlib import Path

# Verified Single-Task LR2e-5 run record
SINGLE_TASK_REFERENCE_RUN_RECORD_PATH = Path(
    "/content/drive/MyDrive/MARBERTv2_LR2e5_COMPLETE/"
    "runs/exp_marbertv2-lr2e5-clean-20261119/"
    "run_record (1).json"
)

# Verify the file exists
assert SINGLE_TASK_REFERENCE_RUN_RECORD_PATH.is_file(), (
    "ERROR: Single-Task run record not found!"
)

# Verify current experiment identity
EXPECTED_MULTI_TASK_RUN_ID = (
    "marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

assert RUN_ID == EXPECTED_MULTI_TASK_RUN_ID, (
    f"Wrong Multi-Task experiment: {RUN_ID}"
)

print("Single-Task reference: FOUND")
print("Single-Task path:", SINGLE_TASK_REFERENCE_RUN_RECORD_PATH)
print("Multi-Task run:", RUN_ID)
print("Ready for comparison.")

In [ ]:
TFIDF_REFERENCE = {
    'complaint_category': 0.7605, 'complaint_urgency': 0.6902,
    'customer_state': 0.7000, 'churn_risk': 0.8629,
}
single_task_reference = None
if SINGLE_TASK_REFERENCE_RUN_RECORD_PATH is not None:
    reference_path = Path(SINGLE_TASK_REFERENCE_RUN_RECORD_PATH)
    single_task_reference = load_run_record(reference_path)
    if (single_task_reference['evaluated_split'] != 'val'
            or single_task_reference['dataset']['version'] != SHARED['dataset_version']
            or single_task_reference['config']['model'] != MODEL_NAME):
        raise ValueError(f'Single-task run record does not match the validation dataset/model: {reference_path}')
    expected_rows = {'train': SHARED['train_rows'], 'val': SHARED['val_rows'], 'test': SHARED['test_rows']}
    if single_task_reference['dataset']['rows'] != expected_rows:
        raise ValueError(f'Single-task run record has different frozen split sizes: {reference_path}')
    reference_seeds = single_task_reference['per_seed']
    if [entry.get('seed') for entry in reference_seeds] != SEEDS:
        raise ValueError(f'Single-task run record must contain exactly the five configured seeds: {reference_path}')
    for entry in reference_seeds:
        if (entry.get('run_id') != single_task_reference['run_id']
                or entry.get('model') != MODEL_NAME or entry.get('split') != 'val'):
            raise ValueError(f'Single-task per-seed identity mismatch: {reference_path}')
    for task in TASKS:
        key = f'{task}_macro_f1'
        if key not in single_task_reference['aggregate']:
            raise ValueError(f'Single-task run record is missing aggregate metric {key}: {reference_path}')
        values = [entry.get('metrics', {}).get(key) for entry in reference_seeds]
        if any(not isinstance(value, (int, float)) or not np.isfinite(value) for value in values):
            raise ValueError(f'Single-task run record has invalid per-seed values for {key}: {reference_path}')
        aggregate = single_task_reference['aggregate'][key]
        if (not np.isclose(aggregate['mean'], np.mean(values), rtol=0, atol=1e-12)
                or not np.isclose(aggregate['std'], np.std(values, ddof=1), rtol=0, atol=1e-12)):
            raise ValueError(f'Single-task aggregate does not match its five seed metrics for {key}: {reference_path}')
    if single_task_reference['config']['other_hyperparameters'].get('model_revision') != MODEL_REVISION:
        raise ValueError(f'Single-task run record uses a different MARBERTv2 revision: {reference_path}')
single_task_means = ({task: single_task_reference['aggregate'][f'{task}_macro_f1']['mean']
                      for task in TASKS} if single_task_reference else {})
single_task_sds = ({task: single_task_reference['aggregate'][f'{task}_macro_f1']['std']
                    for task in TASKS} if single_task_reference else {})
reference_rows = []
for task in TASKS:
    measured = ('aggregate_table' in globals())
    reference_rows.append({
        'task': task, 'single_task_mean': single_task_means.get(task, np.nan),
        'single_task_sd': single_task_sds.get(task, np.nan), 'tfidf': TFIDF_REFERENCE[task],
        'multi_task_mean': (float(aggregate_table.loc[aggregate_table['task'] == task, 'macro_f1_mean'].iloc[0])
                            if measured else np.nan),
        'multi_task_sample_sd': (float(aggregate_table.loc[aggregate_table['task'] == task, 'macro_f1_sample_sd'].iloc[0])
                                 if measured else np.nan),
    })
reference_rows.append({
    'task': 'overall_mean',
    'single_task_mean': float(np.mean(list(single_task_means.values()))) if len(single_task_means) == len(TASKS) else np.nan,
    'single_task_sd': (single_task_reference['aggregate']['combined_macro_f1']['std']
                       if single_task_reference and 'combined_macro_f1' in single_task_reference['aggregate'] else np.nan),
    'tfidf': float(np.mean(list(TFIDF_REFERENCE.values()))),
    'multi_task_mean': combined_mean if 'combined_mean' in globals() else np.nan,
    'multi_task_sample_sd': combined_sd_by_seed if 'combined_sd_by_seed' in globals() else np.nan,
})
reference_table = pd.DataFrame(reference_rows)
reference_table['delta_vs_single_task'] = reference_table['multi_task_mean'] - reference_table['single_task_mean']
reference_table['delta_vs_tfidf'] = reference_table['multi_task_mean'] - reference_table['tfidf']
reference_table['single_task_status'] = 'verified run record' if single_task_reference else 'pending run record'
display(reference_table)
if reference_table['multi_task_mean'].notna().all():
    x = np.arange(len(reference_table))
    width = 0.25
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width, reference_table['tfidf'], width, label='TF-IDF')
    if reference_table['single_task_mean'].notna().all():
        ax.bar(x, reference_table['single_task_mean'], width,
               yerr=reference_table['single_task_sd'], capsize=3, label='MARBERTv2 single-task')
    ax.bar(x + (width if single_task_reference else 0), reference_table['multi_task_mean'], width,
           yerr=reference_table['multi_task_sample_sd'], capsize=3, label='MARBERTv2 multi-task')
    ax.set_xticks(x, reference_table['task'], rotation=20, ha='right')
    ax.set_ylabel('Validation macro-F1')
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    print('Multi-task comparison is pending measured five-seed results; no performance claim is made.')

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

EXPECTED_RUN_ID = "marbertv2-multitask-kendall-lr2e5-confirmation-v1"

assert RUN_ID == EXPECTED_RUN_ID
assert "reference_table" in globals()
assert reference_table["single_task_mean"].notna().all()
assert reference_table["multi_task_mean"].notna().all()

results_dir = RUN_DIR / "RESULTS"
results_dir.mkdir(parents=True, exist_ok=True)

# Save comparison table
csv_path = results_dir / "validation_model_comparison.csv"
reference_table.to_csv(csv_path, index=False, float_format="%.12f")

# Save comparison chart
tasks = reference_table["task"].tolist()
x = range(len(tasks))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))

import numpy as np
x = np.arange(len(tasks))

ax.bar(x - width, reference_table["tfidf"],
       width, label="TF-IDF (reference)")

ax.bar(x, reference_table["single_task_mean"],
       width, yerr=reference_table["single_task_sd"],
       capsize=3, label="MARBERTv2 Single-Task")

ax.bar(x + width, reference_table["multi_task_mean"],
       width, yerr=reference_table["multi_task_sample_sd"],
       capsize=3, label="MARBERTv2 Multi-Task Kendall")

ax.set_xticks(x)
ax.set_xticklabels(tasks, rotation=20, ha="right")
ax.set_ylabel("Validation Macro-F1")
ax.set_ylim(0, 1)
ax.set_title("MARBERTv2 LR2e-5 — Validation Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

png_path = results_dir / "validation_model_comparison.png"
fig.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

print("CSV SAVED:", csv_path, csv_path.is_file())
print("PNG SAVED:", png_path, png_path.is_file())
print("ROWS:", len(reference_table))

## 13. Reproducibility metadata, run record, and prediction export

After all seeds complete, I write one validation prediction file per seed with the existing schema, including probabilities for every task and class but no transcript text. I validate each file by reading it back. Each seed has a loadable `model.safetensors`, tokenizer, and architecture metadata; `latest.pt` is retained separately for optimizer resume. The run record and SHA-256 handoff manifest include paths, file sizes, and hashes. Existing outputs are verified or rejected, never silently overwritten.

In [ ]:
from pathlib import Path

EXPECTED_RUN_ID = "marbertv2-multitask-kendall-lr2e5-confirmation-v1"

assert RUN_ID == EXPECTED_RUN_ID, "Wrong experiment!"

print("RUN_DIR:", RUN_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("RUNS_ROOT:", RUNS_ROOT)

print("\nFive seeds ready:",
      "SEED_RESULTS" in globals()
      and len(SEED_RESULTS) == 5)

print("\nExisting files:")

for filename in [
    "run_record.json",
    "handoff_manifest.json",
    "RESULTS/validation_model_comparison.csv",
    "RESULTS/validation_model_comparison.png",
]:
    path = RUN_DIR / filename
    print(filename, ":", "SAVED" if path.is_file() else "NOT FOUND")

In [ ]:
if 'SEED_RESULTS' in globals() and len(SEED_RESULTS) == len(SEEDS):
    artifact_paths = {'predictions': [], 'metrics': [], 'model_safetensors': [],
                      'architecture': [], 'tokenizers': [], 'optimizer_checkpoints': [],
                      'seed_results': []}
    per_seed_records = []
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    for result in SEED_RESULTS:
        seed = int(result['seed'])
        rows = []
        for index, audio_file in enumerate(VALIDATION_DATASET.audio_files):
            rows.append({'audio_file': audio_file, 'probabilities': {
                task: {label: float(result['probabilities'][task][index][class_index])
                       for class_index, label in enumerate(CLASSES_BY_TASK[task])}
                for task in TASKS
            }})
        path = predictions_path(RUNS_ROOT, RUN_ID, 'val', seed)
        if path.exists():
            loaded = read_predictions_file(path, CLASSES_BY_TASK, expected_split='val')
            expected_meta = {'run_id': RUN_ID, 'model': MODEL_NAME, 'split': 'val',
                             'seed': seed, 'dataset_version': SHARED['dataset_version']}
            if any(loaded['meta'][key] != value for key, value in expected_meta.items()):
                raise ValueError(f'Existing prediction file has a different identity: {path}')
            for row in rows:
                if loaded['probabilities'].get(row['audio_file']) != row['probabilities']:
                    raise FileExistsError(f'Existing prediction file differs for {row["audio_file"]}: {path}')
        else:
            write_predictions_file(path, RUN_ID, MODEL_NAME, 'val', seed,
                                   SHARED['dataset_version'], rows, CLASSES_BY_TASK)
            loaded = read_predictions_file(path, CLASSES_BY_TASK, expected_split='val')
        final_report = evaluate_predictions('val', INPUTS.truth, loaded['harness_rows'],
                                            CLASSES_BY_TASK, EXCLUSIONS)
        if final_report['combined_macro_f1']['mean'] != result['report']['combined_macro_f1']['mean']:
            raise ValueError(f'Exported prediction metrics differ for seed {seed}')
        metric_path = RUN_DIR / f'metrics_val_seed{seed}.json'
        if metric_path.exists():
            stored = json.loads(metric_path.read_text(encoding='utf-8'))
            if stored != result['report']:
                raise FileExistsError(f'Existing metrics differ: {metric_path}')
        else:
            atomic_json_save(result['report'], metric_path)
        seed_dir = RUN_DIR / 'checkpoints' / f'seed_{seed}'
        checkpoint_path = seed_dir / 'model.safetensors'
        architecture_path = seed_dir / 'architecture.json'
        tokenizer_dir = seed_dir / 'tokenizer'
        optimizer_path = seed_dir / 'latest.pt'
        required_artifacts = (checkpoint_path, architecture_path, optimizer_path,
                              tokenizer_dir / 'tokenizer_config.json')
        missing_artifacts = [item for item in required_artifacts if not item.is_file()]
        if missing_artifacts:
            raise FileNotFoundError(f'Seed {seed} is missing handoff/resume artifacts: {missing_artifacts}')
        artifact_paths['predictions'].append(path.relative_to(OUTPUT_ROOT).as_posix())
        artifact_paths['metrics'].append(metric_path.relative_to(OUTPUT_ROOT).as_posix())
        artifact_paths['model_safetensors'].append(checkpoint_path.relative_to(OUTPUT_ROOT).as_posix())
        artifact_paths['architecture'].append(architecture_path.relative_to(OUTPUT_ROOT).as_posix())
        artifact_paths['tokenizers'].append(tokenizer_dir.relative_to(OUTPUT_ROOT).as_posix())
        artifact_paths['optimizer_checkpoints'].append(optimizer_path.relative_to(OUTPUT_ROOT).as_posix())
        artifact_paths['seed_results'].append(
            (RUN_DIR / 'checkpoints' / f'seed_{seed}' / 'validation_result.json').relative_to(OUTPUT_ROOT).as_posix())
        per_seed_records.append(make_seed_result(
            RUN_ID, EXPERIMENT_CONFIG, 'val', seed,
            {**seed_metrics(result['report']['fields']),
             'combined_macro_f1': float(result['report']['combined_macro_f1']['mean'])},
        ))
    record = build_run_record(
        RUN_ID,
        'Measure validation performance of a shared MARBERTv2 encoder with four Kendall-weighted heads',
        EXPERIMENT_CONFIG, SHARED, per_seed_records, 'val', artifact_paths, device_name,
        SEED_RESULTS[0]['report']['exceptions'], ddof=1,
    )
    record_path = RUN_DIR / 'run_record.json'
    if record_path.exists():
        existing_record = load_run_record(record_path)
        for key in ('run_id', 'config', 'evaluated_split', 'per_seed', 'aggregate', 'artifacts', 'exceptions'):
            if existing_record[key] != record[key]:
                raise FileExistsError(f'Existing run record differs in {key}: {record_path}')
    else:
        save_run_record(record, RUNS_ROOT)
    manifest_files = [RUN_DIR / 'config.json', record_path]
    for artifact_group in artifact_paths.values():
        for relative_path in artifact_group:
            artifact_path = OUTPUT_ROOT / relative_path
            if artifact_path.is_dir():
                manifest_files.extend(path for path in sorted(artifact_path.iterdir()) if path.is_file())
            else:
                manifest_files.append(artifact_path)
    unique_manifest_files = sorted(set(manifest_files))
    manifest = {
        'schema_version': 1, 'run_id': RUN_ID,
        'config_fingerprint': CONFIG_FINGERPRINT,
        'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION,
        'learning_rate': LEARNING_RATE, 'seeds': SEEDS,
        'dataset_version': SHARED['dataset_version'], 'evaluated_split': 'val',
        'input_sha256': INPUT_SHA256,
        'inputs': {
            'train': {'filename': TRAIN_CSV.name, 'size_bytes': TRAIN_CSV.stat().st_size,
                      'sha256': INPUT_SHA256['train']},
            'validation': {'filename': VALIDATION_CSV.name, 'size_bytes': VALIDATION_CSV.stat().st_size,
                           'sha256': INPUT_SHA256['validation']},
            'class_weights': {'filename': CLASS_WEIGHTS_PATH.name, 'size_bytes': CLASS_WEIGHTS_PATH.stat().st_size,
                              'sha256': INPUT_SHA256['class_weights']},
        },
        'files': {path.relative_to(OUTPUT_ROOT).as_posix():
                  {'size_bytes': path.stat().st_size, 'sha256': sha256_file(path)}
                  for path in unique_manifest_files if path.is_file()},
    }
    manifest_path = RUN_DIR / 'handoff_manifest.json'
    if manifest_path.exists():
        existing_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if existing_manifest != manifest:
            raise FileExistsError(f'Existing handoff manifest differs: {manifest_path}')
    else:
        atomic_json_save(manifest, manifest_path)
    for relative_path, details in manifest['files'].items():
        path = OUTPUT_ROOT / relative_path
        if path.stat().st_size != details['size_bytes'] or sha256_file(path) != details['sha256']:
            raise IOError(f'Handoff manifest verification failed for {path}')
    print('Validated validation predictions and run record:', record_path)
    print('Verified SHA-256 handoff manifest:', manifest_path)
    print('Selected uncertainty log variances by seed:',
          {item['seed']: item['log_variances'] for item in SEED_RESULTS})
else:
    print('Export is pending. It requires all five completed seed results.')

In [ ]:
from pathlib import Path
import json
import hashlib

RUN_DIR = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5/"
    "runs/exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5"
)

SEEDS_TO_CHECK = [42, 1337, 2023, 7, 99]

missing = []
verified = []
not_in_manifest = []

def check_file(path, label):
    if path.is_file() and path.stat().st_size > 0:
        verified.append(label)
        print("OK     ", label)
    else:
        missing.append(label)
        print("MISSING", label, "->", path)

print("=== EXPERIMENT IDENTITY ===")

assert RUN_DIR.is_dir(), "Experiment directory not found"

config_path = RUN_DIR / "config.json"
check_file(config_path, "Experiment config")

if config_path.is_file():
    config = json.loads(config_path.read_text(encoding="utf-8"))
    print("Config file readable: YES")

print("\n=== FIVE-SEED ARTIFACTS ===")

for seed in SEEDS_TO_CHECK:
    seed_dir = RUN_DIR / "checkpoints" / f"seed_{seed}"

    for filename in [
        "latest.pt",
        "model.safetensors",
        "architecture.json",
        "validation_result.json",
        "tokenizer/tokenizer_config.json",
        "tokenizer/tokenizer.json",
    ]:
        check_file(
            seed_dir / filename,
            f"Seed {seed}: {filename}"
        )

    check_file(
        RUN_DIR / f"metrics_val_seed{seed}.json",
        f"Seed {seed}: metrics"
    )

print("\n=== EXPERIMENT RECORDS ===")

for filename in [
    "run_record.json",
    "handoff_manifest.json",
    "RESULTS/validation_checkpoint_audit.csv",
    "RESULTS/validation_checkpoint_audit.json",
    "RESULTS/validation_per_task_per_seed.csv",
    "RESULTS/validation_model_comparison.csv",
    "RESULTS/validation_model_comparison.png",
    "marbertv2_multitask_colab.ipynb",
]:
    check_file(RUN_DIR / filename, filename)

print("\n=== MANIFEST INTEGRITY ===")

manifest_path = RUN_DIR / "handoff_manifest.json"

if manifest_path.is_file():
    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    expected_run_id = (
        "marbertv2-multitask-kendall-lr2e5-confirmation-v1"
    )

    assert manifest["run_id"] == expected_run_id, (
        "Wrong experiment in manifest!"
    )

    failed_hashes = []

    for relative_path, details in manifest["files"].items():
        path = OUTPUT_ROOT / relative_path

        if not path.is_file():
            failed_hashes.append((relative_path, "MISSING"))
            continue

        sha = hashlib.sha256()

        with path.open("rb") as f:
            for chunk in iter(
                lambda: f.read(8 * 1024 * 1024), b""
            ):
                sha.update(chunk)

        if (
            sha.hexdigest() != details["sha256"]
            or path.stat().st_size != details["size_bytes"]
        ):
            failed_hashes.append(
                (relative_path, "HASH MISMATCH")
            )

    print("Manifest entries:", len(manifest["files"]))
    print("Integrity failures:", len(failed_hashes))

    for item in failed_hashes:
        print("FAILED:", item)

else:
    failed_hashes = [("manifest", "MISSING")]

print("\n=== FINAL AUDIT SUMMARY ===")
print("Verified required files:", len(verified))
print("Missing required files:", len(missing))
print("Manifest integrity failures:", len(failed_hashes))

if not missing and not failed_hashes:
    print("PASS: Required artifacts exist and manifest hashes match.")
else:
    print("REVIEW REQUIRED: Some artifacts are missing or invalid.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

EXPECTED_RUN_ID = "marbertv2-multitask-kendall-lr2e5-confirmation-v1"

assert RUN_ID == EXPECTED_RUN_ID, "Wrong experiment!"
assert "SEED_RESULTS" in globals(), "Run validation analysis first."

assert len(SEED_RESULTS) == 5, "Expected exactly five seeds."
assert {int(r["seed"]) for r in SEED_RESULTS} == set(SEEDS)

ERROR_DIR = RUN_DIR / "RESULTS" / "error_analysis"
ERROR_DIR.mkdir(parents=True, exist_ok=True)

saved_files = []

for task in TASKS:
    labels = list(CLASSES_BY_TASK[task])
    matrices = []

    for result in SEED_RESULTS:
        seed = int(result["seed"])

        assert result["run_id"] == RUN_ID
        assert result["config_fingerprint"] == CONFIG_FINGERPRINT
        assert result["input_sha256"] == INPUT_SHA256

        report = result["report"]["fields"][task]
        matrix = np.asarray(
            report["confusion_matrix"]["counts"],
            dtype=np.int64
        )

        assert matrix.shape == (len(labels), len(labels))
        assert np.all(matrix >= 0)

        df = pd.DataFrame(
            matrix,
            index=labels,
            columns=labels
        )

        df.index.name = "true_label"
        df.columns.name = "predicted_label"

        path = ERROR_DIR / f"{task}_seed{seed}_confusion_matrix.csv"
        df.to_csv(path)

        saved_files.append(path)
        matrices.append(matrix)

    # Aggregate across five seed evaluations
    pooled = np.sum(matrices, axis=0)

    pooled_df = pd.DataFrame(
        pooled,
        index=labels,
        columns=labels
    )

    pooled_df.index.name = "true_label"
    pooled_df.columns.name = "predicted_label"

    pooled_path = ERROR_DIR / f"{task}_pooled_confusion_matrix.csv"
    pooled_df.to_csv(pooled_path)

    saved_files.append(pooled_path)

    print(
        f"{task}: {len(matrices)} seeds | "
        f"{int(pooled.sum())} pooled predictions"
    )

print("\n=== ERROR ANALYSIS EXPORT ===")
print("Saved files:", len(saved_files))
print("Directory:", ERROR_DIR)

assert len(saved_files) == len(TASKS) * (len(SEEDS) + 1)
assert all(p.is_file() and p.stat().st_size > 0 for p in saved_files)

print("PASS: All confusion matrices saved successfully.")

In [ ]:
import numpy as np
import pandas as pd

assert RUN_ID == "marbertv2-multitask-kendall-lr2e5-confirmation-v1"
assert len(SEED_RESULTS) == len(SEEDS) == 5

error_dir = RUN_DIR / "RESULTS" / "error_analysis"
assert error_dir.is_dir()

rows = []

for result in SEED_RESULTS:
    seed = int(result["seed"])

    for task in TASKS:
        labels = list(CLASSES_BY_TASK[task])
        cm = np.asarray(
            result["report"]["fields"][task]
                  ["confusion_matrix"]["counts"],
            dtype=np.int64
        )

        for i, label in enumerate(labels):
            tp = int(cm[i, i])
            support = int(cm[i, :].sum())
            predicted = int(cm[:, i].sum())

            precision = tp / predicted if predicted else 0.0
            recall = tp / support if support else 0.0

            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall else 0.0
            )

            rows.append({
                "seed": seed,
                "task": task,
                "class": label,
                "support": support,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "false_negatives": support - tp,
                "false_positives": predicted - tp,
            })

per_class_df = pd.DataFrame(rows)

summary_df = (
    per_class_df
    .groupby(["task", "class"], as_index=False)
    .agg(
        precision_mean=("precision", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        f1_sd=("f1", "std"),
        support_per_seed=("support", "first"),
        false_negatives_mean=("false_negatives", "mean"),
        false_positives_mean=("false_positives", "mean"),
    )
    .sort_values(["task", "f1_mean"])
)

# Verify that each class has consistent support across seeds
support_counts = per_class_df.groupby(
    ["task", "class"]
)["support"].nunique()

assert (support_counts == 1).all(), (
    "Class support differs across seeds. Investigate before summarizing."
)

detail_path = error_dir / "per_class_metrics_all_seeds.csv"
summary_path = error_dir / "per_class_metrics_summary.csv"

per_class_df.to_csv(detail_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("=== PER-CLASS ERROR ANALYSIS ===")
print("Detailed rows:", len(per_class_df))
print("Summary rows:", len(summary_df))
print("Saved:", detail_path)
print("Saved:", summary_path)

display(summary_df)

assert detail_path.is_file()
assert summary_path.is_file()

print("PASS: Per-class analysis saved to Drive.")

In [ ]:
print("=== VALIDATION SUPPORT AUDIT ===")

for task in TASKS:
    print(f"\nTASK: {task}")

    for result in SEED_RESULTS:
        seed = result["seed"]
        report = result["report"]["fields"][task]

        cm = np.asarray(
            report["confusion_matrix"]["counts"]
        )

        print(
            f"Seed {seed}: "
            f"evaluated={cm.sum()}, "
            f"validation_rows={len(VALIDATION_DATASET.audio_files)}, "
            f"difference={len(VALIDATION_DATASET.audio_files) - cm.sum()}"
        )

print("\n=== EXCLUSION INFORMATION ===")
print("EXCLUSIONS:", EXCLUSIONS)

print("\n=== REPORT EXCEPTIONS ===")
print(SEED_RESULTS[0]["report"].get("exceptions"))

In [ ]:
print("=== VALIDATION DATASET ===")
print("Dataset type:", type(VALIDATION_DATASET))
print("Number of samples:", len(VALIDATION_DATASET.audio_files))
print("First 3 audio files:", VALIDATION_DATASET.audio_files[:3])

print("\n=== GROUND TRUTH ===")
print("INPUTS.truth type:", type(INPUTS.truth))

if isinstance(INPUTS.truth, dict):
    print("First 3 keys:", list(INPUTS.truth.keys())[:3])

print("\n=== PREDICTIONS ===")
result = SEED_RESULTS[0]

print("Seed:", result["seed"])

for task in TASKS:
    probabilities = np.asarray(result["probabilities"][task])
    print(
        task,
        "| shape:", probabilities.shape,
        "| classes:", CLASSES_BY_TASK[task]
    )

print("\n=== EXCLUSION POLICY ===")
print(EXCLUSIONS)

In [ ]:
print("=== GROUND TRUTH STRUCTURE ===")

print("Total truth records:", len(INPUTS.truth))

first = INPUTS.truth[0]

print("First record type:", type(first))
print("First record:", first)

if isinstance(first, dict):
    print("Available keys:", list(first.keys()))

print("\n=== DATASET ALIGNMENT ===")
print("First audio file:", VALIDATION_DATASET.audio_files[0])
print("First prediction seed:", SEED_RESULTS[0]["seed"])

print("\n=== TASK LABELS ===")
for task in TASKS:
    print(task, ":", CLASSES_BY_TASK[task])

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

EXPECTED_RUN_ID = "marbertv2-multitask-kendall-lr2e5-confirmation-v1"

assert RUN_ID == EXPECTED_RUN_ID
assert len(SEED_RESULTS) == len(SEEDS) == 5

ERROR_DIR = RUN_DIR / "RESULTS" / "error_analysis"
ERROR_DIR.mkdir(parents=True, exist_ok=True)

# Validate unique sample identities and alignment
truth_by_file = {
    row["audio_file"]: row
    for row in INPUTS.truth
}

audio_files = list(VALIDATION_DATASET.audio_files)

assert len(truth_by_file) == len(INPUTS.truth) == len(audio_files)
assert len(set(audio_files)) == len(audio_files)
assert set(audio_files) == set(truth_by_file)

all_rows = []

for result in SEED_RESULTS:
    seed = int(result["seed"])

    assert result["run_id"] == RUN_ID
    assert result["config_fingerprint"] == CONFIG_FINGERPRINT
    assert result["input_sha256"] == INPUT_SHA256

    for task in TASKS:
        labels = list(CLASSES_BY_TASK[task])
        probabilities = np.asarray(
            result["probabilities"][task], dtype=float
        )

        assert probabilities.shape == (len(audio_files), len(labels))
        assert np.isfinite(probabilities).all()

        excluded_labels = set(
            EXCLUSIONS.get(task, {}).get("labels", [])
        )

        for index, audio_file in enumerate(audio_files):
            true_label = truth_by_file[audio_file][task]

            if true_label in excluded_labels:
                continue

            assert true_label in labels, (
                f"Unknown label: {task} / {true_label}"
            )

            predicted_index = int(np.argmax(probabilities[index]))
            predicted_label = labels[predicted_index]

            all_rows.append({
                "seed": seed,
                "task": task,
                "audio_file": audio_file,
                "true_label": true_label,
                "predicted_label": predicted_label,
                "correct": predicted_label == true_label,
                "predicted_probability": float(
                    probabilities[index, predicted_index]
                ),
                "true_label_probability": float(
                    probabilities[index, labels.index(true_label)]
                ),
            })

predictions_df = pd.DataFrame(all_rows)

errors_df = predictions_df.loc[
    ~predictions_df["correct"]
].copy()

repeated_errors_df = (
    errors_df.groupby(["task", "audio_file"], as_index=False)
    .agg(
        error_count=("seed", "nunique"),
        true_label=("true_label", "first"),
        predicted_labels=(
            "predicted_label",
            lambda x: " | ".join(sorted(set(x)))
        ),
        mean_predicted_probability=(
            "predicted_probability", "mean"
        ),
    )
    .sort_values(
        ["error_count", "task"],
        ascending=[False, True]
    )
)

persistent_errors_df = repeated_errors_df[
    repeated_errors_df["error_count"] == len(SEEDS)
].copy()

# Verify predictions reproduce saved confusion matrices
for result in SEED_RESULTS:
    seed = int(result["seed"])

    for task in TASKS:
        labels = list(CLASSES_BY_TASK[task])

        subset = predictions_df[
            (predictions_df["seed"] == seed)
            & (predictions_df["task"] == task)
        ]

        calculated = pd.crosstab(
            pd.Categorical(subset["true_label"], categories=labels),
            pd.Categorical(subset["predicted_label"], categories=labels),
            dropna=False
        ).to_numpy()

        original = np.asarray(
            result["report"]["fields"][task]
                  ["confusion_matrix"]["counts"]
        )

        assert np.array_equal(calculated, original), (
            f"Confusion matrix mismatch: {task}, seed {seed}"
        )

# Save analysis
paths = {
    "predictions": ERROR_DIR / "validation_predictions_all_seeds.csv",
    "errors": ERROR_DIR / "misclassified_examples_all_seeds.csv",
    "repeated": ERROR_DIR / "repeated_errors_by_sample.csv",
    "persistent": ERROR_DIR / "persistent_errors_all_five_seeds.csv",
}

predictions_df.to_csv(paths["predictions"], index=False)
errors_df.to_csv(paths["errors"], index=False)
repeated_errors_df.to_csv(paths["repeated"], index=False)
persistent_errors_df.to_csv(paths["persistent"], index=False)

print("=== MISCLASSIFICATION ANALYSIS ===")
print("Evaluated predictions:", len(predictions_df))
print("Misclassified predictions:", len(errors_df))
print("Unique task/sample errors:", len(repeated_errors_df))
print("Persistent errors (5/5 seeds):", len(persistent_errors_df))

print("\n=== ERRORS BY TASK ===")
display(
    errors_df.groupby(["task", "seed"])
    .size()
    .unstack()
)

print("\n=== MOST REPEATED ERRORS ===")
display(repeated_errors_df.head(20))

print("\n=== SAVED FILES ===")
for name, path in paths.items():
    assert path.is_file()
    print(name, ":", path)

print("\nPASS: Misclassification analysis saved to Drive.")

In [ ]:
print("=== VALIDATION TEXT STRUCTURE ===")

print("VALIDATION_DATASET attributes:")
print([
    name for name in vars(VALIDATION_DATASET)
    if not name.startswith("_")
])

print("\n=== VALIDATION CSV ===")
print("Path:", VALIDATION_CSV)

validation_preview = pd.read_csv(
    VALIDATION_CSV,
    nrows=3
)

print("Columns:", validation_preview.columns.tolist())
display(validation_preview)

print("\n=== TARGET FILES ===")
target_files = [
    "frustrated_0030.wav",
    "frustrated_0066.wav",
    "frustrated_0443.wav",
]

print(
    "All targets in ground truth:",
    all(f in truth_by_file for f in target_files)
)

In [ ]:
print("=== PERSISTENT CHURN ERRORS — TEXT REVIEW ===")

target_files = [
    "frustrated_0030.wav",
    "frustrated_0066.wav",
    "frustrated_0443.wav",
]

validation_df = pd.read_csv(VALIDATION_CSV)

assert validation_df["audio_file"].is_unique
assert set(target_files).issubset(set(validation_df["audio_file"]))

review_columns = [
    "audio_file",
    "text",
    "text_whisper",
    "text_norm",
    "complaint_category",
    "complaint_urgency",
    "customer_state",
    "churn_risk",
    "note",
    "confidence",
]

review_df = (
    validation_df.loc[
        validation_df["audio_file"].isin(target_files),
        review_columns
    ]
    .set_index("audio_file")
    .loc[target_files]
    .reset_index()
)

for _, row in review_df.iterrows():
    audio_file = row["audio_file"]

    print("\n" + "=" * 75)
    print("AUDIO FILE:", audio_file)
    print("=" * 75)

    print("\nORIGINAL TEXT:")
    print(row["text"])

    print("\nWHISPER TEXT:")
    print(row["text_whisper"])

    print("\nNORMALIZED TEXT:")
    print(row["text_norm"])

    print("\nGROUND TRUTH:")
    for task in TASKS:
        print(f"{task}: {row[task]}")

    print("\nANNOTATION NOTE:", row["note"])
    print("ANNOTATION CONFIDENCE:", row["confidence"])

    print("\nMODEL PREDICTIONS:")
    subset = predictions_df.loc[
        (predictions_df["audio_file"] == audio_file)
        & (predictions_df["task"] == "churn_risk"),
        [
            "seed",
            "predicted_label",
            "predicted_probability",
            "true_label_probability",
        ],
    ].sort_values("seed")

    print(subset.to_string(index=False))

print("\nPASS: Three persistent churn errors retrieved for review.")

In [ ]:
print("=== ALL AT-RISK VALIDATION CASES ===")

at_risk_df = validation_df.loc[
    validation_df["churn_risk"] == "at_risk"
].copy()

assert len(at_risk_df) == 7

churn_predictions = predictions_df.loc[
    predictions_df["task"] == "churn_risk"
].copy()

summary = (
    churn_predictions.loc[
        churn_predictions["audio_file"].isin(
            at_risk_df["audio_file"]
        )
    ]
    .groupby("audio_file", as_index=False)
    .agg(
        correct_seeds=("correct", "sum"),
        mean_at_risk_probability=(
            "true_label_probability", "mean"
        ),
    )
)

summary["missed_seeds"] = len(SEEDS) - summary["correct_seeds"]

review = at_risk_df[
    ["audio_file", "text", "text_whisper", "text_norm",
     "complaint_category", "complaint_urgency"]
].merge(
    summary,
    on="audio_file",
    validate="one_to_one"
)

assert len(review) == 7

review = review.sort_values(
    ["correct_seeds", "audio_file"]
)

for _, row in review.iterrows():
    print("\n" + "=" * 65)
    print("FILE:", row["audio_file"])
    print("CORRECT SEEDS:", int(row["correct_seeds"]), "/ 5")
    print("MISSED SEEDS:", int(row["missed_seeds"]))
    print(
        "MEAN AT-RISK PROBABILITY:",
        round(row["mean_at_risk_probability"], 4)
    )
    print("CATEGORY:", row["complaint_category"])
    print("URGENCY:", row["complaint_urgency"])
    print("ORIGINAL TEXT:", row["text"])
    print("WHISPER TEXT:", row["text_whisper"])
    print("NORMALIZED TEXT:", row["text_norm"])

print("\nPASS: All seven at-risk cases reviewed.")

In [ ]:
import pandas as pd

assert len(review) == 7

churn_case_summary = review[
    [
        "audio_file",
        "complaint_category",
        "complaint_urgency",
        "correct_seeds",
        "missed_seeds",
        "mean_at_risk_probability",
    ]
].copy()

churn_case_summary["true_label"] = "at_risk"
churn_case_summary["total_seeds"] = len(SEEDS)

churn_case_summary["detection_rate"] = (
    churn_case_summary["correct_seeds"] / len(SEEDS)
)

output_path = (
    RUN_DIR
    / "RESULTS"
    / "error_analysis"
    / "churn_at_risk_case_analysis.csv"
)

churn_case_summary.to_csv(output_path, index=False)

saved = pd.read_csv(output_path)

assert len(saved) == 7
assert saved["audio_file"].is_unique
assert saved["correct_seeds"].sum() == 18
assert saved["missed_seeds"].sum() == 17

print("=== CHURN ERROR ANALYSIS EXPORT ===")
print("Saved:", output_path)
print("Cases:", len(saved))
print("Correct predictions:", saved["correct_seeds"].sum(), "/ 35")
print("Missed predictions:", saved["missed_seeds"].sum(), "/ 35")

print("\nPASS: Churn case analysis saved to Drive.")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

# ============================================================
# MARBERTv2 MULTITASK — ROOT CAUSE & REMEDIATION REPORT
# READ-ONLY INPUTS | NEW REPORT OUTPUTS ONLY
# ============================================================

EXPECTED_RUN_ID = (
    "marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

assert RUN_ID == EXPECTED_RUN_ID

SOURCE_DIR = RUN_DIR / "RESULTS" / "error_analysis"
REPORT_DIR = SOURCE_DIR / "research_report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

assert len(SEEDS) == 5
assert len(SEED_RESULTS) == 5

# ------------------------------------------------------------
# 1. LOAD VERIFIED ANALYSIS ARTIFACTS
# ------------------------------------------------------------

per_class = pd.read_csv(
    SOURCE_DIR / "per_class_metrics_summary.csv"
)

predictions = pd.read_csv(
    SOURCE_DIR / "validation_predictions_all_seeds.csv"
)

errors = pd.read_csv(
    SOURCE_DIR / "misclassified_examples_all_seeds.csv"
)

repeated = pd.read_csv(
    SOURCE_DIR / "repeated_errors_by_sample.csv"
)

persistent = pd.read_csv(
    SOURCE_DIR / "persistent_errors_all_five_seeds.csv"
)

churn_cases = pd.read_csv(
    SOURCE_DIR / "churn_at_risk_case_analysis.csv"
)

validation = pd.read_csv(VALIDATION_CSV)

assert len(predictions) == 20055
assert len(errors) == 3701
assert len(churn_cases) == 7
assert len(persistent) == 510

assert validation["audio_file"].is_unique
assert set(churn_cases["audio_file"]).issubset(
    set(validation["audio_file"])
)

# ------------------------------------------------------------
# 2. TASK-LEVEL ERROR STATISTICS
# ------------------------------------------------------------

task_stats = (
    predictions.groupby("task")
    .agg(
        evaluations=("correct", "size"),
        correct=("correct", "sum")
    )
    .reset_index()
)

task_stats["errors"] = (
    task_stats["evaluations"] - task_stats["correct"]
)

task_stats["error_rate"] = (
    task_stats["errors"] / task_stats["evaluations"]
)

# ------------------------------------------------------------
# 3. CLASS-LEVEL WEAKNESSES
# ------------------------------------------------------------

weak_classes = per_class.sort_values(
    ["f1_mean", "recall_mean"],
    ascending=[True, True]
).copy()

# ------------------------------------------------------------
# 4. PERSISTENT ERROR STATISTICS
# ------------------------------------------------------------

persistent_stats = (
    persistent.groupby("task")
    .size()
    .rename("persistent_errors")
    .reset_index()
)

task_stats = task_stats.merge(
    persistent_stats,
    on="task",
    how="left"
)

task_stats["persistent_errors"] = (
    task_stats["persistent_errors"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# 5. CHURN CASE EVIDENCE
# ------------------------------------------------------------

churn_text = churn_cases.merge(
    validation[
        [
            "audio_file",
            "text",
            "text_whisper",
            "text_norm",
            "complaint_category",
            "complaint_urgency",
            "customer_state",
        ]
    ],
    on="audio_file",
    how="left",
    suffixes=("", "_validation"),
    validate="one_to_one"
)

assert churn_text["text"].notna().all()

churn_text = churn_text.sort_values(
    ["correct_seeds", "audio_file"]
)

# ------------------------------------------------------------
# 6. HYPOTHESES — NOT PROVEN CAUSES
# ------------------------------------------------------------

hypotheses = [
    {
        "id": "H01",
        "issue": "Minority-class imbalance in churn detection",
        "evidence": (
            "Only 7 at-risk validation samples versus "
            "996 not-at-risk samples."
        ),
        "hypothesis": (
            "Limited minority-class representation may "
            "reduce sensitivity to churn indicators."
        ),
        "verification": (
            "Inspect training class counts, compare "
            "per-class learning behavior, and evaluate "
            "controlled class-balancing interventions."
        ),
        "remediation": (
            "Consider targeted annotation, minority-class "
            "sampling, or loss adjustments. Change one "
            "factor at a time."
        ),
        "priority": "HIGH",
        "status": "UNVERIFIED_HYPOTHESIS",
    },
    {
        "id": "H02",
        "issue": "Context-dependent churn detection",
        "evidence": (
            "Three persistent false negatives occur in "
            "billing_and_charges, including explicit "
            "switching or cancellation language."
        ),
        "hypothesis": (
            "The model may under-recognize churn signals "
            "when embedded in billing complaints."
        ),
        "verification": (
            "Audit training examples by churn label and "
            "complaint category; compare matched examples "
            "with similar churn expressions."
        ),
        "remediation": (
            "Collect and annotate diverse billing-related "
            "churn examples, then run controlled experiments."
        ),
        "priority": "HIGH",
        "status": "UNVERIFIED_HYPOTHESIS",
    },
    {
        "id": "H03",
        "issue": "Potential annotation ambiguity",
        "evidence": (
            "frustrated_0030.wav is labeled at_risk but "
            "does not contain an explicit intention to leave."
        ),
        "hypothesis": (
            "The annotation policy may include implicit "
            "trust-loss indicators, creating a broader "
            "definition than explicit cancellation intent."
        ),
        "verification": (
            "Review original annotation guidelines and "
            "obtain blinded independent annotation."
        ),
        "remediation": (
            "Clarify the operational definition of at_risk "
            "and adjudicate ambiguous cases without using "
            "model predictions as ground truth."
        ),
        "priority": "HIGH",
        "status": "UNVERIFIED_HYPOTHESIS",
    },
    {
        "id": "H04",
        "issue": "High-urgency false positives",
        "evidence": (
            "High urgency has mean precision approximately "
            "0.410 and recall approximately 0.781."
        ),
        "hypothesis": (
            "The classifier may favor recall over precision "
            "for high urgency."
        ),
        "verification": (
            "Inspect confusion pairs, decision margins, "
            "training distribution, and label criteria."
        ),
        "remediation": (
            "Evaluate validation-only threshold or "
            "decision-rule adjustments against a "
            "predefined precision-recall objective."
        ),
        "priority": "MEDIUM",
        "status": "UNVERIFIED_HYPOTHESIS",
    },
    {
        "id": "H05",
        "issue": "Confusion between emotional states",
        "evidence": (
            "mad has mean precision approximately 0.383 "
            "and recall approximately 0.848."
        ),
        "hypothesis": (
            "Expressions of frustration may overlap "
            "linguistically with anger."
        ),
        "verification": (
            "Review true/predicted confusion pairs and "
            "conduct blinded annotation review."
        ),
        "remediation": (
            "Refine class definitions and test targeted "
            "examples with consistent annotation."
        ),
        "priority": "MEDIUM",
        "status": "UNVERIFIED_HYPOTHESIS",
    },
    {
        "id": "H06",
        "issue": "Potential ASR sensitivity",
        "evidence": (
            "Some original and Whisper transcripts differ, "
            "but the exact model input column has not yet "
            "been independently confirmed."
        ),
        "hypothesis": (
            "Transcription errors could influence "
            "predictions for certain complaints."
        ),
        "verification": (
            "Confirm the tokenization input column and "
            "perform a controlled ASR-input ablation."
        ),
        "remediation": (
            "Improve transcription quality only if a "
            "controlled experiment demonstrates benefit."
        ),
        "priority": "MEDIUM",
        "status": "UNVERIFIED_HYPOTHESIS",
    },
]

hypothesis_df = pd.DataFrame(hypotheses)

# ------------------------------------------------------------
# 7. REMEDIATION EXPERIMENT PLAN
# ------------------------------------------------------------

experiment_plan = [
    {
        "experiment": "E01",
        "change": "Audit training minority-class counts",
        "type": "DATA_AUDIT",
        "primary_metric": "Class support and label consistency",
        "constraint": "No training or test-set changes",
    },
    {
        "experiment": "E02",
        "change": "Independent churn-label adjudication",
        "type": "ANNOTATION_AUDIT",
        "primary_metric": "Inter-annotator agreement",
        "constraint": "Review without revealing model predictions",
    },
    {
        "experiment": "E03",
        "change": "Churn class-balancing strategy",
        "type": "CONTROLLED_TRAINING",
        "primary_metric": "At-risk recall, precision, F1, PR-AUC",
        "constraint": (
            "Fixed splits, five seeds, and all other "
            "hyperparameters"
        ),
    },
    {
        "experiment": "E04",
        "change": "Validation-only churn threshold selection",
        "type": "DECISION_RULE",
        "primary_metric": (
            "At-risk recall with predefined "
            "false-positive constraint"
        ),
        "constraint": (
            "No test-set tuning; report validation "
            "selection optimism"
        ),
    },
    {
        "experiment": "E05",
        "change": "Targeted billing-context augmentation",
        "type": "CONTROLLED_TRAINING",
        "primary_metric": (
            "At-risk recall by complaint category"
        ),
        "constraint": (
            "Training-only additions; preserve held-out "
            "validation and test sets"
        ),
    },
    {
        "experiment": "E06",
        "change": "Original-versus-ASR text ablation",
        "type": "INPUT_ABLATION",
        "primary_metric": "Macro F1 and per-class recall",
        "constraint": (
            "Confirm baseline text pipeline first; "
            "hold other factors fixed"
        ),
    },
]

plan_df = pd.DataFrame(experiment_plan)

# ------------------------------------------------------------
# 8. BUILD RESEARCH REPORT
# ------------------------------------------------------------

def md_table(df):
    return df.to_markdown(index=False, floatfmt=".4f")

task_view = task_stats[
    [
        "task",
        "evaluations",
        "errors",
        "error_rate",
        "persistent_errors",
    ]
]

weak_view = weak_classes[
    [
        "task",
        "class",
        "support_per_seed",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "f1_sd",
    ]
]

churn_view = churn_text[
    [
        "audio_file",
        "complaint_category",
        "correct_seeds",
        "missed_seeds",
        "mean_at_risk_probability",
    ]
]

report_lines = [
    "# MARBERTv2 Multi-Task — Error Analysis, Root-Cause Hypotheses & Remediation",
    "",
    f"**Run ID:** `{RUN_ID}`",
    f"**Generated (UTC):** {datetime.now(timezone.utc).isoformat()}",
    "**Evaluation:** Validation set only",
    "**Seeds:** 42, 1337, 2023, 7, 99",
    "",
    "## 1. Executive Summary",
    "",
    f"- Total task-level evaluations: {len(predictions):,}.",
    f"- Incorrect task-level evaluations: {len(errors):,}.",
    f"- Unique task/sample error pairs: {len(repeated):,}.",
    f"- Persistent errors across all five seeds: {len(persistent):,}.",
    "- Repeated seed evaluations are not independent samples.",
    "- Root causes are not established by observational error analysis alone.",
    "",
    "## 2. Task-Level Error Statistics",
    "",
    md_table(task_view),
    "",
    "## 3. Per-Class Performance",
    "",
    md_table(weak_view),
    "",
    "## 4. Churn Risk — Seven Positive Cases",
    "",
    "The validation set contains seven at-risk cases. "
    "Their detection outcomes are:",
    "",
    md_table(churn_view),
    "",
    "### Case-Level Observations",
    "",
    "- `frustrated_0030.wav`: billing complaint with loss of trust "
    "but no explicit cancellation or switching statement.",
    "- `frustrated_0066.wav`: explicitly considers changing provider; "
    "missed by all five seeds.",
    "- `frustrated_0443.wav`: explicitly considers stopping service; "
    "missed by all five seeds.",
    "- `frustrated_0241.wav`: conditional switching statement; "
    "correct in three of five seeds.",
    "- `calm_0246.wav`: direct switching inquiry; "
    "correct in all five seeds.",
    "- `frustrated_0416.wav`: network-related switching intent; "
    "correct in all five seeds.",
    "- `mad_0031.wav`: service disruption and explicit leaving intent; "
    "correct in all five seeds.",
    "",
    "### Interpretation",
    "",
    "The observed errors suggest that churn detection may vary "
    "by complaint context. This is an exploratory hypothesis, "
    "not evidence of a confirmed causal mechanism.",
    "",
    "## 5. Root-Cause Hypotheses",
    "",
]

for h in hypotheses:
    report_lines.extend([
        f"### {h['id']} — {h['issue']}",
        "",
        f"**Observed evidence:** {h['evidence']}",
        "",
        f"**Hypothesis:** {h['hypothesis']}",
        "",
        f"**How to verify:** {h['verification']}",
        "",
        f"**Proposed remediation:** {h['remediation']}",
        "",
        f"**Priority:** {h['priority']}",
        "",
    ])

report_lines.extend([
    "## 6. Controlled Remediation Plan",
    "",
    md_table(plan_df),
    "",
    "## 7. Evaluation Integrity & Limitations",
    "",
    "- Customer-state label `unclear` has zero training examples "
    "and one validation example; it is excluded from "
    "customer-state scoring only.",
    "- The churn-positive validation support is only seven, "
    "so recall estimates are statistically unstable.",
    "- Model probabilities have not been demonstrated to be calibrated.",
    "- Existing validation analysis is exploratory and cannot "
    "establish generalization to unseen data.",
    "- Thresholds or model configurations must not be selected "
    "using independent test results.",
    "- Proposed changes require separately identified experiment "
    "runs with immutable baseline artifacts.",
    "",
    "## 8. Recommended Next Steps",
    "",
    "1. Verify the training text column and class distributions.",
    "2. Audit ambiguous annotations independently.",
    "3. Freeze the baseline and define evaluation objectives.",
    "4. Run one-factor-at-a-time experiments under fixed splits.",
    "5. Evaluate the selected final configuration once on "
    "the independent test set.",
    "",
    "## 9. Conclusion",
    "",
    "The validation results identify persistent, operationally "
    "relevant false negatives in churn-risk detection, "
    "including complaints with explicit switching and "
    "service-cancellation language. The evidence motivates "
    "targeted investigation but does not independently prove "
    "class imbalance, context bias, annotation inconsistency, "
    "or ASR errors as causal explanations.",
])

report_text = "\n".join(report_lines) + "\n"

# ------------------------------------------------------------
# 9. SAVE REPORT ARTIFACTS
# ------------------------------------------------------------

report_md = REPORT_DIR / "MARBERTv2_Error_Analysis_Report.md"
report_json = REPORT_DIR / "MARBERTv2_Error_Analysis_Report.json"
hypotheses_csv = REPORT_DIR / "root_cause_hypotheses.csv"
plan_csv = REPORT_DIR / "remediation_experiment_plan.csv"
task_csv = REPORT_DIR / "task_error_statistics.csv"
churn_csv = REPORT_DIR / "churn_case_evidence.csv"

report_md.write_text(report_text, encoding="utf-8")

report_json.write_text(
    json.dumps(
        {
            "run_id": RUN_ID,
            "evaluation_split": "validation",
            "summary": {
                "evaluations": len(predictions),
                "errors": len(errors),
                "unique_task_sample_errors": len(repeated),
                "persistent_errors": len(persistent),
            },
            "task_statistics": task_stats.to_dict("records"),
            "hypotheses": hypotheses,
            "experiment_plan": experiment_plan,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

hypothesis_df.to_csv(hypotheses_csv, index=False)
plan_df.to_csv(plan_csv, index=False)
task_stats.to_csv(task_csv, index=False)
churn_text.to_csv(churn_csv, index=False)

# ------------------------------------------------------------
# 10. VERIFY OUTPUTS
# ------------------------------------------------------------

outputs = [
    report_md,
    report_json,
    hypotheses_csv,
    plan_csv,
    task_csv,
    churn_csv,
]

for path in outputs:
    assert path.exists() and path.stat().st_size > 0

with open(report_json, encoding="utf-8") as f:
    saved_report = json.load(f)

assert saved_report["run_id"] == RUN_ID
assert saved_report["summary"]["errors"] == 3701

print("=== RESEARCH REPORT EXPORT ===")
print("Report directory:", REPORT_DIR)
print("Generated files:", len(outputs))

for path in outputs:
    print("OK:", path.name)

print("\nPASS: Research report and remediation plan saved to Drive.")

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import pandas as pd

# ============================================================
# MARBERTv2 — FINAL HANDOFF AUDIT (READ ONLY)
# ============================================================

EXPECTED_RUN_ID = (
    "marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

EXPECTED_ROOT = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5"
)

EXPECTED_RUN_DIR = (
    EXPECTED_ROOT
    / "runs"
    / f"exp_{EXPECTED_RUN_ID}"
)

assert RUN_ID == EXPECTED_RUN_ID
assert RUN_DIR.resolve() == EXPECTED_RUN_DIR.resolve()

root = EXPECTED_RUN_DIR
results = root / "RESULTS"
errors = results / "error_analysis"
reports = errors / "research_report"

required_files = {
    "Experiment records": [
        root / "run_record.json",
        root / "handoff_manifest.json",
    ],
    "Validation results": [
        results / "validation_checkpoint_audit.csv",
        results / "validation_checkpoint_audit.json",
        results / "validation_per_task_per_seed.csv",
        results / "validation_model_comparison.csv",
        results / "validation_model_comparison.png",
    ],
    "Error analysis": [
        errors / "per_class_metrics_all_seeds.csv",
        errors / "per_class_metrics_summary.csv",
        errors / "validation_predictions_all_seeds.csv",
        errors / "misclassified_examples_all_seeds.csv",
        errors / "repeated_errors_by_sample.csv",
        errors / "persistent_errors_all_five_seeds.csv",
        errors / "churn_at_risk_case_analysis.csv",
    ],
    "Research report": [
        reports / "MARBERTv2_Error_Analysis_Report.md",
        reports / "MARBERTv2_Error_Analysis_Report.json",
        reports / "root_cause_hypotheses.csv",
        reports / "remediation_experiment_plan.csv",
        reports / "task_error_statistics.csv",
        reports / "churn_case_evidence.csv",
    ],
}

print("=== FINAL HANDOFF AUDIT ===")
print("Experiment:", EXPECTED_RUN_ID)
print("Folder:", root)

missing = []
empty = []
verified = []

for group, paths in required_files.items():
    print(f"\n=== {group.upper()} ===")

    for path in paths:
        if not path.is_file():
            print("MISSING:", path.name)
            missing.append(str(path))
        elif path.stat().st_size == 0:
            print("EMPTY:", path.name)
            empty.append(str(path))
        else:
            print(
                "OK:",
                path.name,
                f"({path.stat().st_size:,} bytes)"
            )
            verified.append(str(path))

# Check five seed model artifacts without guessing folder names
print("\n=== MODEL ARTIFACT DISCOVERY ===")

seed_dirs = {}

for seed in [42, 1337, 2023, 7, 99]:
    candidates = [
        p for p in root.rglob("latest.pt")
        if f"seed{seed}" in str(p).lower()
        or f"seed_{seed}" in str(p).lower()
    ]

    seed_dirs[seed] = candidates

    print(f"Seed {seed}: {len(candidates)} checkpoint candidate(s)")

    for p in candidates:
        print("  ", p.relative_to(root))

# Inspect report data
print("\n=== REPORT DATA INTEGRITY ===")

pred_path = errors / "validation_predictions_all_seeds.csv"
err_path = errors / "misclassified_examples_all_seeds.csv"
persistent_path = errors / "persistent_errors_all_five_seeds.csv"

if all(p.is_file() for p in [
    pred_path, err_path, persistent_path
]):
    pred = pd.read_csv(pred_path)
    err = pd.read_csv(err_path)
    persistent = pd.read_csv(persistent_path)

    print("Predictions:", len(pred))
    print("Errors:", len(err))
    print("Persistent errors:", len(persistent))

    assert len(pred) == 20055
    assert len(err) == 3701
    assert len(persistent) == 510

    print("PASS: Analysis row counts match previous results.")

print("\n=== FINAL SUMMARY ===")
print("Verified required files:", len(verified))
print("Missing:", len(missing))
print("Empty:", len(empty))

if not missing and not empty:
    print("PASS: All listed handoff files exist and are nonempty.")
else:
    print("ATTENTION: Some files need investigation.")

print(
    "\nNOTE: This is an existence and basic consistency audit, "
    "not yet a complete SHA-256 integrity verification."
)

In [ ]:
import json

manifest_path = RUN_DIR / "handoff_manifest.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("=== MANIFEST STRUCTURE INSPECTION ===")
print("Manifest type:", type(manifest).__name__)

if isinstance(manifest, dict):
    print("Top-level keys:", list(manifest.keys()))

    for key, value in manifest.items():
        print(f"\nKEY: {key}")
        print("TYPE:", type(value).__name__)

        if isinstance(value, list):
            print("ENTRIES:", len(value))
            if value:
                print(
                    "FIRST ENTRY:",
                    json.dumps(
                        value[0],
                        ensure_ascii=False,
                        indent=2,
                        default=str
                    )[:1500]
                )

        elif isinstance(value, dict):
            print("SUBKEYS:", list(value.keys())[:15])

        else:
            print("VALUE:", str(value)[:300])

print("\nPASS: Manifest structure inspected without modifications.")

In [ ]:
from pathlib import Path
import hashlib
import json

# ============================================================
# MARBERTv2 — ORIGINAL MANIFEST SHA-256 VERIFICATION
# READ ONLY — NO FILES MODIFIED
# ============================================================

project_root = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5"
)

run_dir = project_root / "runs" / (
    "exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

manifest_path = run_dir / "handoff_manifest.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

assert manifest["run_id"] == (
    "marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

assert manifest["config_fingerprint"] == (
    "55d74f49ed44d55a0ebef67eb57f0c4d755d3f7dd65bc5aa71e8f58f8ae87f3f"
)

assert set(manifest["seeds"]) == {42, 1337, 2023, 7, 99}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

files = manifest["files"]

print("=== ORIGINAL MANIFEST INTEGRITY AUDIT ===")
print("Manifest entries:", len(files))

# Inspect the actual stored hash representation.
first_value = next(iter(files.values()))

print("Stored entry type:", type(first_value).__name__)
print("Stored entry sample:", str(first_value)[:500])

verified = []
missing = []
mismatched = []
unsupported = []

for relative_path, metadata in files.items():

    path = project_root / relative_path

    # Ensure manifest entries stay inside the project folder.
    if not path.resolve().is_relative_to(
        project_root.resolve()
    ):
        unsupported.append((relative_path, "unsafe path"))
        continue

    if not path.is_file():
        missing.append(relative_path)
        continue

    if isinstance(metadata, str):
        expected_hash = metadata

    elif isinstance(metadata, dict):
        expected_hash = (
            metadata.get("sha256")
            or metadata.get("SHA256")
        )

    else:
        expected_hash = None

    if not isinstance(expected_hash, str):
        unsupported.append(
            (relative_path, "unrecognized hash metadata")
        )
        continue

    expected_hash = expected_hash.lower().strip()

    if (
        len(expected_hash) != 64
        or any(c not in "0123456789abcdef"
               for c in expected_hash)
    ):
        unsupported.append(
            (relative_path, "invalid SHA-256 value")
        )
        continue

    actual_hash = sha256_file(path)

    if actual_hash == expected_hash:
        verified.append(relative_path)
    else:
        mismatched.append(relative_path)

print("\n=== INTEGRITY RESULTS ===")
print("Verified:", len(verified))
print("Missing:", len(missing))
print("Hash mismatches:", len(mismatched))
print("Unsupported entries:", len(unsupported))

for name, entries in [
    ("MISSING", missing),
    ("MISMATCHED", mismatched),
    ("UNSUPPORTED", unsupported),
]:
    if entries:
        print(f"\n{name}:")
        for entry in entries:
            print(" -", entry)

if (
    len(verified) == len(files)
    and not missing
    and not mismatched
    and not unsupported
):
    print(
        "\nPASS: Every original manifest file "
        "matches its recorded SHA-256."
    )
else:
    print(
        "\nATTENTION: Manifest integrity "
        "verification is incomplete or failed."
    )

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

RUN_DIR = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5/"
    "runs/exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

print("Drive connected:", Path("/content/drive/MyDrive").exists())
print("Experiment folder exists:", RUN_DIR.is_dir())
print("Manifest exists:", (RUN_DIR / "handoff_manifest.json").is_file())
print("Research report exists:", (
    RUN_DIR / "RESULTS/error_analysis/research_report/"
    "MARBERTv2_Error_Analysis_Report.md"
).is_file())

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

RUN_DIR = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5/"
    "runs/exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

results = RUN_DIR / "RESULTS"
errors = results / "error_analysis"
reports = errors / "research_report"

files_to_register = [
    RUN_DIR / "run_record.json",
    RUN_DIR / "handoff_manifest.json",

    results / "validation_checkpoint_audit.csv",
    results / "validation_checkpoint_audit.json",
    results / "validation_per_task_per_seed.csv",
    results / "validation_model_comparison.csv",
    results / "validation_model_comparison.png",

    errors / "per_class_metrics_all_seeds.csv",
    errors / "per_class_metrics_summary.csv",
    errors / "validation_predictions_all_seeds.csv",
    errors / "misclassified_examples_all_seeds.csv",
    errors / "repeated_errors_by_sample.csv",
    errors / "persistent_errors_all_five_seeds.csv",
    errors / "churn_at_risk_case_analysis.csv",

    reports / "MARBERTv2_Error_Analysis_Report.md",
    reports / "MARBERTv2_Error_Analysis_Report.json",
    reports / "root_cause_hypotheses.csv",
    reports / "remediation_experiment_plan.csv",
    reports / "task_error_statistics.csv",
    reports / "churn_case_evidence.csv",
]

def calculate_sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert len(files_to_register) == 20
assert len(set(files_to_register)) == 20

records = {}

for path in files_to_register:
    assert path.is_file(), f"Missing: {path}"
    assert path.stat().st_size > 0, f"Empty: {path}"

    relative = path.relative_to(RUN_DIR).as_posix()

    records[relative] = {
        "size_bytes": path.stat().st_size,
        "sha256": calculate_sha256(path),
    }

manifest = {
    "schema_version": 1,
    "run_id": "marbertv2-multitask-kendall-lr2e5-confirmation-v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Supplementary validation and research analysis integrity",
    "original_manifest_preserved": True,
    "files": records,
}

output = RUN_DIR / "supplementary_integrity_manifest.json"

# Preserve an existing supplementary manifest rather than
# silently overwriting a previous integrity record.
if output.exists():
    raise FileExistsError(
        f"Supplementary manifest already exists: {output}"
    )

output.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

with output.open("r", encoding="utf-8") as f:
    saved = json.load(f)

assert len(saved["files"]) == 20

for relative, metadata in saved["files"].items():
    path = RUN_DIR / relative
    assert path.stat().st_size == metadata["size_bytes"]
    assert calculate_sha256(path) == metadata["sha256"]

print("=== SUPPLEMENTARY MANIFEST ===")
print("Saved:", output)
print("Registered files:", len(saved["files"]))
print("Verified hashes:", len(saved["files"]))
print("Original manifest unchanged: YES")
print("\nPASS: Supplementary integrity manifest saved and verified.")

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5"
)

print("=== CONFIG FILES ===")
for p in sorted(ROOT.rglob("*config*.json")):
    if p.is_file():
        print(p.relative_to(ROOT))

print("\n=== ARCHITECTURE FILES ===")
for p in sorted(ROOT.rglob("architecture.json")):
    if p.is_file():
        print(p.relative_to(ROOT))

print("\n=== NOTEBOOK FILES ===")
for p in sorted(ROOT.rglob("*.ipynb")):
    if p.is_file():
        print(
            p.relative_to(ROOT),
            "| Size:", p.stat().st_size,
            "bytes"
        )

print("\nPASS: Config and notebook inventory completed.")

In [ ]:
from pathlib import Path
import json

RUN_DIR = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5/"
    "runs/exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

notebook_path = RUN_DIR / "marbertv2_multitask_colab.ipynb"
config_path = RUN_DIR / "config.json"

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

with config_path.open("r", encoding="utf-8") as f:
    config = json.load(f)

cells = notebook.get("cells", [])
code_cells = [
    cell for cell in cells
    if cell.get("cell_type") == "code"
]

notebook_text = "\n".join(
    "".join(cell.get("source", []))
    for cell in code_cells
)

checks = {
    "Error analysis": "per_class_metrics_summary.csv",
    "Prediction export": "validation_predictions_all_seeds.csv",
    "Churn case analysis": "churn_at_risk_case_analysis.csv",
    "Research report": "MARBERTv2_Error_Analysis_Report.md",
    "Supplementary manifest": "supplementary_integrity_manifest.json",
}

print("=== SAVED NOTEBOOK AUDIT ===")
print("Total cells:", len(cells))
print("Code cells:", len(code_cells))
print("Size:", notebook_path.stat().st_size, "bytes")

for label, marker in checks.items():
    print(
        f"{label}:",
        "FOUND" if marker in notebook_text else "NOT FOUND"
    )

print("\n=== EXPERIMENT CONFIG ===")
print("Config keys:", list(config.keys()))

print("\n=== FILE MODIFICATION TIME ===")
from datetime import datetime, timezone

print(
    datetime.fromtimestamp(
        notebook_path.stat().st_mtime,
        tz=timezone.utc
    ).isoformat()
)

print("\nPASS: Saved notebook and config inspected.")

In [ ]:
from google.colab import _message

print("=== CURRENT COLAB NOTEBOOK ===")

try:
    notebook = _message.blocking_request(
        "get_ipynb",
        timeout_sec=30
    )["ipynb"]

    cells = notebook.get("cells", [])

    source = "\n".join(
        "".join(cell.get("source", []))
        for cell in cells
    )

    checks = {
        "Error analysis": "per_class_metrics_summary.csv",
        "Prediction export": "validation_predictions_all_seeds.csv",
        "Churn analysis": "churn_at_risk_case_analysis.csv",
        "Research report": "MARBERTv2_Error_Analysis_Report.md",
        "Supplementary manifest": "supplementary_integrity_manifest.json",
    }

    print("Total cells:", len(cells))

    for name, marker in checks.items():
        print(
            name + ":",
            "FOUND" if marker in source else "NOT FOUND"
        )

except Exception as e:
    print("Could not inspect current notebook:", repr(e))

In [ ]:
from google.colab import _message
from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib

RUN_DIR = Path(
    "/content/drive/MyDrive/MARBERTv2_MULTITASK_LR2e5/"
    "runs/exp_marbertv2-multitask-kendall-lr2e5-confirmation-v1"
)

# Retrieve the current open notebook
response = _message.blocking_request(
    "get_ipynb",
    timeout_sec=60
)

notebook = response["ipynb"]

assert isinstance(notebook, dict)
assert len(notebook.get("cells", [])) >= 117

# Keep the original notebook untouched
output = RUN_DIR / "marbertv2_multitask_COMPLETE_HANDOFF.ipynb"

if output.exists():
    raise FileExistsError(
        f"File already exists; refusing to overwrite: {output}"
    )

# Validate notebook structure before saving
assert notebook.get("nbformat") == 4
assert all(
    cell.get("cell_type") in ("code", "markdown", "raw")
    for cell in notebook["cells"]
)

# Save the new copy
with output.open("x", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=1)

# Reopen and verify
with output.open("r", encoding="utf-8") as f:
    saved = json.load(f)

assert len(saved["cells"]) == len(notebook["cells"])

saved_source = "\n".join(
    "".join(c.get("source", []))
    for c in saved["cells"]
)

required_markers = [
    "per_class_metrics_summary.csv",
    "validation_predictions_all_seeds.csv",
    "churn_at_risk_case_analysis.csv",
    "MARBERTv2_Error_Analysis_Report.md",
    "supplementary_integrity_manifest.json",
]

for marker in required_markers:
    assert marker in saved_source, f"Missing marker: {marker}"

digest = hashlib.sha256(output.read_bytes()).hexdigest()

print("=== COMPLETE NOTEBOOK HANDOFF ===")
print("Saved:", output)
print("Cells:", len(saved["cells"]))
print("Size:", output.stat().st_size, "bytes")
print("SHA-256:", digest)
print("Original notebook preserved: YES")
print("Required analysis markers: 5/5 PASS")
print("\nPASS: Complete notebook saved and verified.")

## 14. Final handoff

1. Upload this notebook to Colab, choose a GPU runtime, and mount Drive. In the Drive setup cell, set `PROJECT_ROOT` to the exact folder containing `phase2-code/` and the frozen inputs. For a Shared drive, use its mounted `/content/drive/Shareddrives/...` path. Set `DATA_ROOT` to the exact directory containing the three frozen input files, initially `PROJECT_ROOT/frozen-v2`.
2. Run the environment and Drive setup cells. Confirm all required project modules import from `CODE_ROOT` and `TRAINING_CODE`, and each explicit train, validation, and weights path exists. Do not add a test path.
3. Run the frozen input checks, label mapping checks, tokenizer coverage, and model setup. Run the one-batch smoke test. Continue only if the weighted losses, four output shapes, gradients, uncertainty update, safetensors round trip, and prediction schema checks pass.
4. Full training is not automatic. After the smoke test succeeds, set only `RUN_FIVE_SEED_TRAINING = True` in the training cell and run it. Keep the fixed run ID, learning rate 3e-5, seeds, batch size, and epoch budget unchanged.
5. If Colab disconnects, rerun setup and smoke checks, keep the same run ID and configuration, set `RESUME = True`, then run the training cell. It rejects changed input hashes, configuration, runtime identity, or incompatible checkpoints. A completed run is not overwritten.
6. After all five seeds complete, run the validation, aggregation, baseline comparison, and export cells. Verify `model.safetensors`, tokenizer, `architecture.json`, `latest.pt`, validation predictions and metrics for every seed, `run_record.json`, and the verified `handoff_manifest.json` under `marbertv2-multitask-output/runs/exp_<run_id>/`.
7. Single-task comparison remains pending unless an exact verified validation run record is configured. The held-out test is never loaded or evaluated. Keep Drive artifacts private and do not copy model checkpoints into Git.